# 02. Canonical Item Schema and Frozen Item Evidence — Herbal Supplements

This notebook converts heterogeneous item metadata into the category-specific schema used by downstream retrieval and analysis. It parses the structured `details` field, constructs the primary Brand field from `store` and mapped metadata, normalizes category-specific values, and maps them into shared semantic roles: category or product type, form, ingredient or composition, need or benefit, claim or constraint, target context, and sensory evidence. Brand remains a separate preference facet.

The production item representation separates catalog evidence from frozen review-derived item signals. Catalog evidence contains titles, descriptions, feature text, Brand, and normalized functional facets. Review-derived signals are extracted only from review titles and bodies dated strictly before 31 March 2022 at 23:56:10.358 UTC. They are aggregated at parent-item level into bounded phrase families; raw review text is not exported. Ratings, helpfulness, sentiment, and language-model outputs are not used to construct these signals.

Brand is retained in retrieval and profile-safe representations but marked as unavailable to synthetic-query construction. Frozen review-derived signals enter retrieval item text and graph inputs but remain excluded from the user-profile source text.

The notebook also computes rating, verified-purchase, helpful-vote, and review-length aggregates as intermediate diagnostics. These fields are not admitted to the production item documents or item graph created by Notebook 04. The stored execution exports 27,407 parent-item rows together with the shared-role schema, long-form facet table, facet vocabulary, coverage diagnostics, and provenance manifests.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==== Load Libraries and Set Display Options ====
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import ast
import json
import os
import platform
import re
import socket

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

print("Libraries loaded.")


Libraries loaded.


In [3]:
# ==== Define Inputs, Outputs, and Frozen Cutoff ====
CATEGORY_ID = "herbal"
CATEGORY_FOLDER = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
STAGE = "stage0_feature_engineering"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER

PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data/processed/items"
PROCESSED_REVIEWS_DIR = PROJECT_ROOT / "data/processed/reviews"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / STAGE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ITEMS_PATH = PROJECT_ROOT / "data" / "raw" / "items_Herbal_Supplements_W2_2019_2022.parquet"
REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Herbal_Supplements_W2_2019_2022.parquet"

SCHEMA_BASE_PATH = PROCESSED_ITEMS_DIR / "items_Herbal_Supplements_W2_2019_2022_schema_base.parquet"
SCHEMA_OUTPUT_PATH = PROCESSED_ITEMS_DIR / "herbal_item_schema.parquet"
SCHEMA_FULL_OUTPUT_PATH = PROCESSED_ITEMS_DIR / "herbal_item_schema_full.parquet"
COMMON_ITEM_SCHEMA_PATH = PROCESSED_ITEMS_DIR / "herbal_item_schema_common.parquet"
ITEM_FACETS_PATH = PROCESSED_ITEMS_DIR / "herbal_items_facets.parquet"
FACET_VOCAB_PATH = PROCESSED_ITEMS_DIR / "herbal_facet_vocab.parquet"
COLUMN_SUMMARY_PATH = OUTPUT_DIR / "herbal_item_schema_column_summary.csv"
COVERAGE_SUMMARY_PATH = OUTPUT_DIR / "herbal_item_schema_coverage_summary.csv"
QC_PREVIEW_PATH = OUTPUT_DIR / "herbal_item_schema_qc_preview.csv"
RUN_MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"
CONFIG_SNAPSHOT_PATH = OUTPUT_DIR / "config_snapshot.yaml"
RESULTS_OVERALL_PATH = OUTPUT_DIR / "results_overall.csv"
DIAGNOSTICS_SUMMARY_PATH = OUTPUT_DIR / "diagnostics_summary.csv"
COMMON_SCHEMA_ROLE_MAP_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_role_map.csv"
COMMON_SCHEMA_COVERAGE_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_role_coverage.csv"
COMMON_SCHEMA_UNIFORMITY_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_uniformity_summary.csv"
COMMON_SCHEMA_CONTRACT_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_contract.json"
COMMON_SCHEMA_DIAGNOSTICS_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_diagnostics.csv"
BRAND_POLICY_SUMMARY_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_brand_policy_summary.csv"
COMMON_SCHEMA_COVERAGE_NO_BRAND_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_schema_role_coverage_no_brand.csv"

for directory in [PROCESSED_ITEMS_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "project_root": str(PROJECT_ROOT),
    "processed_items_dir": str(PROCESSED_ITEMS_DIR),
    "processed_reviews_dir": str(PROCESSED_REVIEWS_DIR),
    "output_dir": str(OUTPUT_DIR),
    "random_seed": 42,
    "qc_preview_rows": 500,
    "downstream_contract": "notebooks_03_04_05_harmonized_item_schema",
}

REQUIRED_INPUT_PATHS = {
    "items_raw": ITEMS_PATH,
    "reviews_raw": REVIEWS_PATH,
}

EVALUATION_WINDOW_MONTHS = 9
EVALUATION_WINDOW_END = pd.Timestamp("2022-12-31T23:56:10.358000+00:00")
EVALUATION_WINDOW_START = (
    EVALUATION_WINDOW_END - pd.DateOffset(months=EVALUATION_WINDOW_MONTHS)
)
TRAIN_REVIEW_CUTOFF_EXCLUSIVE = EVALUATION_WINDOW_START

REVIEW_REPUTATION_SOURCE = "historical_reviews_before_train_cutoff"
REVIEW_REPUTATION_MAX_PHRASES_PER_FAMILY = 12

RATING_USED_FOR_REVIEW_REPUTATION = False
SENTIMENT_USED_FOR_REVIEW_REPUTATION = False
LLM_USED_FOR_REVIEW_REPUTATION = False
RAW_REVIEW_TEXT_EXPORTED_FOR_REVIEW_REPUTATION = False

missing_inputs = {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ITEMS_PATH:", ITEMS_PATH)
print("REVIEWS_PATH:", REVIEWS_PATH)
print("SCHEMA_BASE_PATH:", SCHEMA_BASE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements
ITEMS_PATH: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/items_Herbal_Supplements_W2_2019_2022.parquet
REVIEWS_PATH: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/raw/reviews_Herbal_Supplements_W2_2019_2022.parquet
SCHEMA_BASE_PATH: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/items_Herbal_Supplements_W2_2019_2022_schema_base.parquet
OUTPUT_DIR: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_feature_engineering


In [4]:
# ==== Load Raw Item and Review Tables ====
items = pd.read_parquet(ITEMS_PATH)
reviews = pd.read_parquet(REVIEWS_PATH)

print("items shape :", items.shape)
print("reviews shape:", reviews.shape)

display(items.head(3))
display(reviews.head(3))

items shape : (27407, 12)
reviews shape: (593363, 9)


,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),4.6,375,"[""Value Pack 2 Bottles of Bio-Curcumin Elite 400 mg, 60 Vegetarian Capsules (2x60)"", ""Up to 7 times more absorbable than conventional Curcumin supplements"", ""During the summer months products may arrive warm but Amazon stores and ships products in accordance with manufacturers' recommendations, ...","[""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioavailability and tissue distribution of free curcuminoids than unformulated curcumin, and they last much longer in the bloodstream. That translat...",46.98,Life Extension,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Curcumin""]","{""Item Form"": ""capsules"", ""Brand"": ""Life Extension"", ""Age Range (Description)"": ""Adult"", ""Material Feature"": ""Vegetarian"", ""Recommended Uses For Product"": ""Healthy Inflammatory"", ""Is Discontinued By Manufacturer"": ""No"", ""Product Dimensions"": ""2 x 4 x 2 inches; 5.1 Ounces"", ""Item model number"": ""...",B01LYS06EF,None
1,Health & Personal Care,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces…",3.9,95,"[""The flavor is somewhat peppery and slightly sweet, with a strong and spicy aroma."", ""Apart from being a great flavoring agent, ginger root also makes a great daily supplement."", ""The Ginger Root Extract Liquid is made of Ginger root liquid extract, purified Water and Propylene glycol."", ""The s...",[],None,Naturevibe Botanicals,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Ginger""]","{""Item Form"": ""Liquid"", ""Brand"": ""Naturevibe Botanicals"", ""Age Range (Description)"": ""Adult"", ""Material Feature"": ""Liquid"", ""Number of Items"": ""1"", ""Package Dimensions"": ""5 x 3.78 x 2.32 inches; 2 Ounces"", ""Date First Available"": ""January 15, 2021"", ""Manufacturer"": ""Naturevibe Botanicals""}",B08T67YDQF,None
2,Health & Personal Care,"Nature's Way Artichoke, 60 Capsules (Pack of 2)",4.2,43,[],[],None,Nature's Way,"[""Health & Household"", ""Vitamins, Minerals & Supplements"", ""Herbal Supplements"", ""Artichoke""]","{""Item Form"": ""Capsule"", ""Brand"": ""Nature's Way"", ""Age Range (Description)"": ""Adult"", ""Recommended Uses For Product"": ""Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.*"", ""Number of Items"": ""2"", ""Is Discontinued By Manufacturer"": ""No"", ""Product Dimensions"": ""...",B002LIMQQA,None


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,BLOCKS IRON ABSORPTION...BE AWARE!,"Although this WAS one of my favorite products for keeping the immune system up and reducing allergies, it blocks iron absorption. Caffeine and calcium also block iron absorption. If you take this, be sure your FERRITIN levels are very high...as some people become symptomatic (severe anxiety, rap...",B0019LWTQW,B07C1XKDBV,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1593364174638,176,True
1,5.0,I CRAVE this stuff!,Blends nicely (I use my frother stick) with No grit or residue. Tastes fresh. ( I have tried others that tasted so swampy that I couldn't drink them. This one does not. I actually look forward to drinking it nice and cold!,B01N33C6EP,B09W7GK6WL,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1654227757388,4,True
2,5.0,drops,Thought I would try these. I am surprised how little you use. Great product. I'll be back to get more if I ever use all of this.,B01IF9UBW4,B09GTZSDF2,AEKDF2ANHCJWJQSPSMPV5TNTNZJA,1595802315366,0,True


In [5]:
# ==== Define Parsing and Normalization Helpers ====
def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x != "" else None

def normalize_text_lower(x):
    x = normalize_text(x)
    return x.lower() if x is not None else None

def parse_maybe_dict(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            return ast.literal_eval(x)
        except Exception:
            try:
                return json.loads(x)
            except Exception:
                return None
    return None

def parse_maybe_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            val = ast.literal_eval(x)
            if isinstance(val, list):
                return val
        except Exception:
            try:
                val = json.loads(x)
                if isinstance(val, list):
                    return val
            except Exception:
                return []
    return []

def safe_join_list(x, sep=" | "):
    vals = parse_maybe_list(x)
    vals = [str(v).strip() for v in vals if normalize_text(v) is not None]
    return sep.join(vals) if vals else None

def get_last_category(x):
    vals = parse_maybe_list(x)
    if len(vals) == 0:
        return None
    return normalize_text(vals[-1])

def get_category_path_text(x):
    vals = parse_maybe_list(x)
    vals = [str(v).strip() for v in vals if normalize_text(v) is not None]
    return " > ".join(vals) if vals else None

def get_category_depth(x):
    return len(parse_maybe_list(x))

def get_detail_value(details_obj, keys):
    d = parse_maybe_dict(details_obj)
    if not isinstance(d, dict):
        return None
    for k in keys:
        if k in d:
            return normalize_text(d[k])
    return None

def clean_token_text(x):
    x = normalize_text(x)
    if x is None:
        return None
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def canonicalize_simple(x):
    x = clean_token_text(x)
    if x is None:
        return None
    return x.lower()

def safe_nunique(series):
    try:
        return series.nunique(dropna=True)
    except TypeError:
        normalized = series.map(
            lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True)
            if isinstance(x, (list, dict))
            else x
        )
        return normalized.nunique(dropna=True)

In [6]:
# ==== Parse Structured Metadata ====
items_work = items.copy()

items_work["details_parsed"] = items_work["details"].apply(parse_maybe_dict)
items_work["categories_parsed"] = items_work["categories"].apply(parse_maybe_list)

items_work["category_path_text"] = items_work["categories"].apply(get_category_path_text)
items_work["category_depth"] = items_work["categories"].apply(get_category_depth)
items_work["sub_category"] = items_work["categories"].apply(get_last_category)

print("Parsed details non-null:", items_work["details_parsed"].notna().sum())
print("Parsed categories non-null:", items_work["categories_parsed"].map(len).gt(0).sum())

preview_cols = [
    "parent_asin",
    "title",
    "store",
    "brand_from_details",
    "sub_category",
    "item_form",
    "recommended_use",
    "specific_uses",
    "product_benefits",
    "active_ingredients",
    "special_ingredients",
    "diet_type",
    "flavor",
]
preview_cols = [col for col in preview_cols if col in items_work.columns]

display(items_work[preview_cols].head(10))

Parsed details non-null: 27407
Parsed categories non-null: 27407


,parent_asin,title,store,sub_category
0,B01LYS06EF,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),Life Extension,Curcumin
1,B08T67YDQF,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces…",Naturevibe Botanicals,Ginger
2,B002LIMQQA,"Nature's Way Artichoke, 60 Capsules (Pack of 2)",Nature's Way,Artichoke
3,B0B1236V1Q,"Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free",Mushroom Revival,Mushrooms
4,B00TQYN2Q0,"RYUKAKUSAN Herbal Drop, Yuzu, 11 Count",Ryukakusan,Alfalfa
5,B008JDWTS6,"Frontier Herb Milk Thistle, 1 Pound",Frontier,Milk Thistle
6,B08HHXGRWK,"Spore Metabolic Boost - Intelligent Functional Mushroom Formulations | 60 Vegan Capsules (30 Day Supply) Cordyceps, Maitake | Immune Support, Energy & Metabolism Boost",Spore,Mushrooms
7,B00DPMZZAA,"Top Secret Nutrition Concentrated Red Palm Oil Diet Supplement, 60 Count",Top Secret Nutrition,Herbal Supplements
8,B075B2XFXN,"PMS Secret Alcohol-Free, Glycerite Black Cohosh, Cramp, Vitex, Valerian, Dandelion, Chamomile, St. John's Wort. Tincture, Herbal Extract Hormonal Imbalance Support 4 OZ",Secrets of the Tribe,Herbal Supplements
9,B005UYXWS8,"Natures Way Neem Leaf 100 Vegetable capsule, 100 ct",Nature's Way,Neem


In [7]:
# ==== Inspect Details-Key Coverage ====
detail_key_counter = Counter()

for d in items_work["details_parsed"].dropna():
    if isinstance(d, dict):
        detail_key_counter.update(d.keys())

detail_key_df = pd.DataFrame(
    detail_key_counter.most_common(150),
    columns=["detail_key", "count"]
)

display(detail_key_df.head(50))

,detail_key,count
0,Manufacturer,25540
1,Date First Available,24163
2,Brand,21482
3,Item Form,20066
4,Age Range (Description),18256
5,Is Discontinued By Manufacturer,13508
6,Material Feature,13309
7,Package Dimensions,11668
8,Product Dimensions,11478
9,Number of Items,9865


In [8]:
# ==== Map Herbal Metadata Keys ====
DETAIL_KEY_MAP = {
    "brand_from_details": ["Brand", "Brand Name"],
    "item_form": ["Item Form", "Dosage Form"],
    "age_range": ["Age Range (Description)"],
    "material_feature": ["Material Feature"],
    "recommended_use": ["Recommended Uses For Product", "Use for"],
    "specific_uses": ["Specific Uses For Product"],
    "product_benefits": ["Product Benefits"],
    "active_ingredients": ["Active Ingredients"],
    "special_ingredients": ["Special Ingredients"],
    "primary_supplement_type": ["Primary Supplement Type"],
    "diet_type": ["Diet Type"],
    "flavor": ["Flavor"],
    "scent": ["Scent"],
    "unit_count": ["Unit Count"],
    "number_of_items": ["Number of Items"],
    "package_dimensions": ["Package Dimensions", "Product Dimensions"],
    "manufacturer": ["Manufacturer"],
    "date_first_available": ["Date First Available"],
    "item_model_number": ["Item model number"],
    "is_discontinued": ["Is Discontinued By Manufacturer"],
}

In [9]:
# ==== Expand Mapped Metadata Fields ====
for new_col, key_list in DETAIL_KEY_MAP.items():
    items_work[new_col] = items_work["details_parsed"].apply(lambda x: get_detail_value(x, key_list))

preview_cols = [
    "parent_asin",
    "title",
    "store",
    "brand_from_details",
    "sub_category",
    "item_form",
    "recommended_use",
    "specific_uses",
    "product_benefits",
    "active_ingredients",
    "special_ingredients",
    "diet_type",
    "flavor",
]

display(items_work[preview_cols].head(10))

,parent_asin,title,store,brand_from_details,sub_category,item_form,recommended_use,specific_uses,product_benefits,active_ingredients,special_ingredients,diet_type,flavor
0,B01LYS06EF,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),Life Extension,Life Extension,Curcumin,capsules,Healthy Inflammatory,None,None,None,None,None,None
1,B08T67YDQF,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces…",Naturevibe Botanicals,Naturevibe Botanicals,Ginger,Liquid,None,None,None,None,None,None,None
2,B002LIMQQA,"Nature's Way Artichoke, 60 Capsules (Pack of 2)",Nature's Way,Nature's Way,Artichoke,Capsule,Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.*,None,None,None,None,None,None
3,B0B1236V1Q,"Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free",Mushroom Revival,Mushroom Revival,Mushrooms,Drop,None,None,None,None,None,None,Reishi Calm
4,B00TQYN2Q0,"RYUKAKUSAN Herbal Drop, Yuzu, 11 Count",Ryukakusan,Ryukakusan,Alfalfa,Drops,None,Cough,None,"Anhydrous,chamomile,eucalyptus,jujube,lemon,licorice",None,None,None
5,B008JDWTS6,"Frontier Herb Milk Thistle, 1 Pound",Frontier,None,Milk Thistle,None,None,None,None,None,None,None,None
6,B08HHXGRWK,"Spore Metabolic Boost - Intelligent Functional Mushroom Formulations | 60 Vegan Capsules (30 Day Supply) Cordyceps, Maitake | Immune Support, Energy & Metabolism Boost",Spore,Spore,Mushrooms,Capsules,"Immune Support,Boost Energy",None,None,None,None,None,None
7,B00DPMZZAA,"Top Secret Nutrition Concentrated Red Palm Oil Diet Supplement, 60 Count",Top Secret Nutrition,Top Secret Nutrition,Herbal Supplements,Softgel,None,None,Weight Loss Support,None,None,None,None
8,B075B2XFXN,"PMS Secret Alcohol-Free, Glycerite Black Cohosh, Cramp, Vitex, Valerian, Dandelion, Chamomile, St. John's Wort. Tincture, Herbal Extract Hormonal Imbalance Support 4 OZ",Secrets of the Tribe,Secrets of the Tribe,Herbal Supplements,Drop,None,None,None,None,None,Gluten Free,None
9,B005UYXWS8,"Natures Way Neem Leaf 100 Vegetable capsule, 100 ct",Nature's Way,Nature's Way,Neem,Leaf,Dietary Supplement,None,None,None,None,None,None


In [10]:
# ==== Construct the Primary Brand Field ====
brand_source = (
    items_work["store"]
    if "store" in items_work.columns
    else pd.Series(pd.NA, index=items_work.index)
)

items_work["brand"] = brand_source
items_work["brand_primary"] = items_work["brand"].fillna(items_work["brand_from_details"])

if "manufacturer" in items_work.columns:
    items_work["brand_primary"] = items_work["brand_primary"].fillna(items_work["manufacturer"])

items_work["brand"] = items_work["brand"].apply(clean_token_text)
items_work["brand_primary"] = items_work["brand_primary"].apply(clean_token_text)

brand_match_df = items_work[
    items_work["brand"].notna() & items_work["brand_from_details"].notna()
].copy()

brand_match_df["brand_match"] = (
    brand_match_df["brand"].astype(str).str.strip().str.lower()
    ==
    brand_match_df["brand_from_details"].astype(str).str.strip().str.lower()
)

print("Rows with both brand sources:", len(brand_match_df))
print("Exact brand match ratio:", brand_match_df["brand_match"].mean() if len(brand_match_df) > 0 else None)

Rows with both brand sources: 21486
Exact brand match ratio: 0.9835241552638928


In [11]:
# ==== Normalize Core Metadata Fields ====
FORM_MAP = {
    "capsules": "capsule",
    "capsule": "capsule",
    "tablet": "tablet",
    "tablets": "tablet",
    "powder": "powder",
    "liquid": "liquid",
    "drop": "drops",
    "drops": "drops",
    "gummy": "gummy",
    "gummies": "gummy",
    "softgel": "softgel",
    "softgels": "softgel",
    "tea bags": "tea_bag",
    "tea bag": "tea_bag",
}

YES_NO_MAP = {
    "yes": "yes",
    "true": "yes",
    "no": "no",
    "false": "no",
}

def normalize_form(x):
    x = canonicalize_simple(x)
    if x is None:
        return None
    return FORM_MAP.get(x, x)

def normalize_yes_no(x):
    x = canonicalize_simple(x)
    if x is None:
        return None
    return YES_NO_MAP.get(x, x)

def normalize_sub_category(x):
    x = clean_token_text(x)
    if x is None:
        return None
    return x.lower()

def normalize_brand(x):
    x = clean_token_text(x)
    if x is None:
        return None
    return x.lower()

items_work["brand_norm"] = items_work["brand_primary"].apply(normalize_brand)
items_work["sub_category_norm"] = items_work["sub_category"].apply(normalize_sub_category)
items_work["item_form_norm"] = items_work["item_form"].apply(normalize_form)
items_work["is_discontinued_norm"] = items_work["is_discontinued"].apply(normalize_yes_no)

display(
    items_work[
        ["parent_asin", "brand_primary", "brand_norm", "sub_category", "sub_category_norm", "item_form", "item_form_norm"]
    ].head(10)
)

,parent_asin,brand_primary,brand_norm,sub_category,sub_category_norm,item_form,item_form_norm
0,B01LYS06EF,Life Extension,life extension,Curcumin,curcumin,capsules,capsule
1,B08T67YDQF,Naturevibe Botanicals,naturevibe botanicals,Ginger,ginger,Liquid,liquid
2,B002LIMQQA,Nature's Way,nature's way,Artichoke,artichoke,Capsule,capsule
3,B0B1236V1Q,Mushroom Revival,mushroom revival,Mushrooms,mushrooms,Drop,drops
4,B00TQYN2Q0,Ryukakusan,ryukakusan,Alfalfa,alfalfa,Drops,drops
5,B008JDWTS6,Frontier,frontier,Milk Thistle,milk thistle,None,None
6,B08HHXGRWK,Spore,spore,Mushrooms,mushrooms,Capsules,capsule
7,B00DPMZZAA,Top Secret Nutrition,top secret nutrition,Herbal Supplements,herbal supplements,Softgel,softgel
8,B075B2XFXN,Secrets of the Tribe,secrets of the tribe,Herbal Supplements,herbal supplements,Drop,drops
9,B005UYXWS8,Nature's Way,nature's way,Neem,neem,Leaf,leaf


In [12]:
# ==== Build Category-Specific Facet Families ====
items_work["family_brand"] = items_work["brand_norm"]
items_work["family_sub_category"] = items_work["sub_category_norm"]
items_work["family_item_form"] = items_work["item_form_norm"]

items_work["family_benefit_function"] = items_work["product_benefits"].fillna(items_work["specific_uses"])
items_work["family_benefit_function"] = items_work["family_benefit_function"].apply(clean_token_text)

items_work["family_usage_need"] = items_work["recommended_use"].fillna(items_work["specific_uses"])
items_work["family_usage_need"] = items_work["family_usage_need"].apply(clean_token_text)

items_work["family_ingredient_composition"] = (
    items_work["active_ingredients"]
    .fillna(items_work["special_ingredients"])
    .fillna(items_work["primary_supplement_type"])
)
items_work["family_ingredient_composition"] = items_work["family_ingredient_composition"].apply(clean_token_text)

items_work["family_claims_diet"] = items_work["diet_type"].fillna(items_work["material_feature"])
items_work["family_claims_diet"] = items_work["family_claims_diet"].apply(clean_token_text)

items_work["family_flavor"] = items_work["flavor"].fillna(items_work["scent"])
items_work["family_flavor"] = items_work["family_flavor"].apply(clean_token_text)

items_work["family_usage_target"] = items_work["age_range"].apply(clean_token_text)

display(
    items_work[
        [
            "parent_asin", "family_brand", "family_sub_category", "family_item_form",
            "family_benefit_function", "family_usage_need", "family_ingredient_composition",
            "family_claims_diet", "family_flavor", "family_usage_target"
        ]
    ].head(15)
)

,parent_asin,family_brand,family_sub_category,family_item_form,family_benefit_function,family_usage_need,family_ingredient_composition,family_claims_diet,family_flavor,family_usage_target
0,B01LYS06EF,life extension,curcumin,capsule,None,Healthy Inflammatory,None,Vegetarian,None,Adult
1,B08T67YDQF,naturevibe botanicals,ginger,liquid,None,None,None,Liquid,None,Adult
2,B002LIMQQA,nature's way,artichoke,capsule,None,Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.*,None,None,None,Adult
3,B0B1236V1Q,mushroom revival,mushrooms,drops,None,None,None,None,Reishi Calm,Adult
4,B00TQYN2Q0,ryukakusan,alfalfa,drops,Cough,Cough,"Anhydrous,chamomile,eucalyptus,jujube,lemon,licorice",None,None,None
5,B008JDWTS6,frontier,milk thistle,None,None,None,None,None,None,None
6,B08HHXGRWK,spore,mushrooms,capsule,None,"Immune Support,Boost Energy",None,"Plant Based, Vegan",None,Adult
7,B00DPMZZAA,top secret nutrition,herbal supplements,softgel,Weight Loss Support,None,None,GMO Free,None,Adult
8,B075B2XFXN,secrets of the tribe,herbal supplements,drops,None,None,None,Gluten Free,None,Adult
9,B005UYXWS8,nature's way,neem,leaf,None,Dietary Supplement,None,Vegetarian,None,Adult


In [13]:
# ==== Build Catalog Text and Evidence Partitions ====
items_work["title_text"] = items_work["title"].fillna("").astype(str).str.strip()
items_work["features_text"] = items_work["features"].fillna("").astype(str).str.strip()
items_work["description_text"] = items_work["description"].fillna("").astype(str).str.strip()


def join_non_empty(parts, sep=" | "):
    vals = []
    for p in parts:
        if p is None:
            continue
        p = str(p).strip()
        if p != "":
            vals.append(p)
    return sep.join(vals) if vals else ""


def join_unique_non_empty(parts, sep=" | "):
    vals = []
    seen = set()
    for part in parts:
        if part is None:
            continue
        if isinstance(part, (list, tuple, set)):
            iter_values = part
        else:
            iter_values = re.split(r"\s*\|\s*", str(part))
        for value in iter_values:
            text = clean_token_text(value)
            if text is None:
                continue
            key = text.lower()
            if key not in seen:
                seen.add(key)
                vals.append(text)
    return sep.join(vals) if vals else ""


FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
PRODUCTION_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED = True
HISTORICAL_REVIEW_REPUTATION_QUARANTINED = False
BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"
IDENTIFIER_POLICY = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

HERBAL_GENERIC_CATEGORY_ANCHORS = {
    "herbal",
    "supplement",
    "herbal supplement",
    "dietary supplement",
}
HERBAL_GENERIC_UTILITY_TOKENS = {
    "support",
    "supports",
    "help",
    "helps",
    "promote",
    "promotes",
    "boost",
    "formula",
    "blend",
    "complex",
    "product",
    "solution",
}
HERBAL_CONTEXT_DEPENDENT_UTILITY_TOKENS = {
    "daily",
    "natural",
    "wellness",
    "health",
    "care",
    "routine",
}
HERBAL_SPECIFIC_PHRASE_EXCEPTIONS = {
    "immune support",
    "digestive support",
    "sleep support",
    "joint support",
    "stress support",
    "energy support",
    "liver support",
}

HARMONIZED_BRAND_SOURCE_COLS = ["family_brand", "brand_primary"]
HARMONIZED_QUERY_SAFE_FACET_SOURCES = {
    "facet_category_text": ["family_sub_category"],
    "facet_form_text": ["family_item_form"],
    "facet_ingredient_text": ["family_ingredient_composition"],
    "facet_benefit_text": ["family_benefit_function", "family_usage_need"],
    "facet_claim_text": ["family_claims_diet"],
    "facet_flavor_text": ["family_flavor"],
    "facet_audience_text": ["family_usage_target"],
}
QUERY_SAFE_FACET_SOURCE_COLUMNS = [
    source_col
    for source_cols in HARMONIZED_QUERY_SAFE_FACET_SOURCES.values()
    for source_col in source_cols
]
FUNCTIONAL_FACET_SOURCE_COLUMNS = ["specific_query_safe_facet_text"]
CORE_TEXT_SOURCE_COLUMNS = ["title_text", "brand_facet_text", "query_safe_facet_text", "description_text", "features_text"]
CORE_SPARSE_TEXT_SOURCE_COLUMNS = ["title_text", "brand_facet_text", "query_safe_facet_text", "features_text"]

PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS = [
    "brand_facet_text",
    "functional_facet_text",
    "title_text",
    "description_text",
]
HISTORICAL_REVIEW_TEXT_COLUMNS = [
    "historical_review_reputation_text",
    "historical_review_ingredient_text",
    "historical_review_benefit_text",
    "historical_review_form_text",
    "historical_review_claim_diet_text",
    "historical_review_flavor_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
]
SOURCE_LISTS_USED_FOR_PRODUCTION_CORE = {
    "query_safe_facet_text": QUERY_SAFE_FACET_SOURCE_COLUMNS,
    "functional_facet_text": FUNCTIONAL_FACET_SOURCE_COLUMNS,
    "canonical_retrieval_text_core": CORE_TEXT_SOURCE_COLUMNS,
    "canonical_text_dense_core": CORE_TEXT_SOURCE_COLUMNS,
    "canonical_text_sparse_core": CORE_SPARSE_TEXT_SOURCE_COLUMNS,
    "profile_source_text_core": PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS,
}


def _join_existing_columns(row, columns):
    return join_unique_non_empty(row.get(col) for col in columns if col in row.index)


items_work["facet_brand_text"] = items_work.apply(
    lambda row: _join_existing_columns(row, HARMONIZED_BRAND_SOURCE_COLS),
    axis=1,
)
items_work["brand_facet_text"] = items_work["facet_brand_text"]

for out_col, source_cols in HARMONIZED_QUERY_SAFE_FACET_SOURCES.items():
    items_work[out_col] = items_work.apply(
        lambda row, cols=source_cols: _join_existing_columns(row, cols),
        axis=1,
    )

def split_facet_values(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    values = []
    for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", str(value)):
        cleaned = clean_token_text(part)
        if cleaned is not None:
            values.append(cleaned)

    return values


BAD_BRAND_FACET_VALUES = (
    HERBAL_GENERIC_CATEGORY_ANCHORS
    | HERBAL_GENERIC_UTILITY_TOKENS
    | HERBAL_CONTEXT_DEPENDENT_UTILITY_TOKENS
)


def remove_bad_brand_values(value):
    kept = [
        part
        for part in split_facet_values(value)
        if part.lower() not in BAD_BRAND_FACET_VALUES
    ]
    return join_unique_non_empty(kept)


items_work["facet_brand_text"] = items_work["facet_brand_text"].map(remove_bad_brand_values)
items_work["brand_facet_text"] = items_work["facet_brand_text"]

bad_brand_norms = {value.lower() for value in BAD_BRAND_FACET_VALUES}
bad_brand_rows = items_work[
    items_work["brand_facet_text"]
    .fillna("")
    .astype(str)
    .map(lambda value: bool({part.lower() for part in split_facet_values(value)} & bad_brand_norms))
]

if len(bad_brand_rows):
    display(bad_brand_rows[["parent_asin", "brand_primary", "brand_facet_text"]].head(20))
    raise RuntimeError("Generic/category/utility tokens leaked into brand_facet_text.")


def remove_row_brand_values(text_value, brand_value):
    brand_norms = {
        value.lower()
        for value in split_facet_values(brand_value)
    }

    kept = [
        value
        for value in split_facet_values(text_value)
        if value.lower() not in brand_norms
    ]

    return join_unique_non_empty(kept)


query_safe_cols = list(HARMONIZED_QUERY_SAFE_FACET_SOURCES.keys())

for col in query_safe_cols:
    items_work[col] = [
        remove_row_brand_values(text_value, brand_value)
        for text_value, brand_value in zip(
            items_work[col],
            items_work["brand_facet_text"],
        )
    ]

items_work["query_safe_facet_text"] = items_work.apply(
    lambda row: _join_existing_columns(row, query_safe_cols),
    axis=1,
)

_generic_anchor_norms = {value.lower() for value in HERBAL_GENERIC_CATEGORY_ANCHORS}
_generic_utility_norms = {value.lower() for value in HERBAL_GENERIC_UTILITY_TOKENS}
_context_utility_norms = {value.lower() for value in HERBAL_CONTEXT_DEPENDENT_UTILITY_TOKENS}
_specific_exception_norms = {value.lower() for value in HERBAL_SPECIFIC_PHRASE_EXCEPTIONS}


def partition_query_safe_text(value):
    generic_anchors = []
    generic_utilities = []
    context_utilities = []
    specific_values = []

    for part in re.split(r"\s*\|\s*", str(value or "")):
        text = clean_token_text(part)
        if text is None:
            continue
        norm = text.lower()
        if norm in _specific_exception_norms:
            specific_values.append(text)
        elif norm in _generic_anchor_norms:
            generic_anchors.append(text)
        elif norm in _generic_utility_norms:
            generic_utilities.append(text)
        elif norm in _context_utility_norms:
            context_utilities.append(text)
        else:
            specific_values.append(text)

    return pd.Series({
        "generic_category_anchor_text": join_unique_non_empty(generic_anchors),
        "generic_utility_token_text": join_unique_non_empty(generic_utilities),
        "context_dependent_utility_token_text": join_unique_non_empty(context_utilities),
        "specific_query_safe_facet_text": join_unique_non_empty(specific_values),
    })


partition_cols = [
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "specific_query_safe_facet_text",
]
items_work = items_work.drop(columns=partition_cols, errors="ignore")
items_work = pd.concat(
    [items_work, items_work["query_safe_facet_text"].apply(partition_query_safe_text)],
    axis=1,
)
items_work["functional_facet_text"] = items_work["specific_query_safe_facet_text"]


def normalized_facet_values(value):
    return {
        part.lower()
        for part in split_facet_values(value)
    }


brand_in_query_safe_source_rows = int(
    items_work.apply(
        lambda row: any(
            bool(
                normalized_facet_values(row.get(col))
                & normalized_facet_values(row.get("brand_facet_text"))
            )
            for col in query_safe_cols
        ),
        axis=1,
    ).sum()
)

if brand_in_query_safe_source_rows:
    raise RuntimeError(
        "Query-safe facet source columns still contain row brand values: "
        f"{brand_in_query_safe_source_rows}"
    )


brand_in_functional_rows = int(
    items_work.apply(
        lambda row: bool(
            normalized_facet_values(row.get("functional_facet_text"))
            & normalized_facet_values(row.get("brand_facet_text"))
        ),
        axis=1,
    ).sum()
)

if brand_in_functional_rows:
    raise RuntimeError(
        "functional_facet_text still contains row brand values: "
        f"{brand_in_functional_rows}"
    )

items_work["review_reputation_facet_text"] = ""
items_work["review_reputation_only_text"] = ""
items_work["identifier_diagnostic_text"] = items_work.apply(
    lambda r: join_unique_non_empty([
        r.get("parent_asin"),
        r.get("manufacturer"),
        r.get("item_model_number"),
    ]),
    axis=1,
)

items_work["canonical_metadata_text"] = items_work.apply(
    lambda r: join_unique_non_empty([
        r.get("title_text"),
        r.get("brand_facet_text"),
        r.get("query_safe_facet_text"),
        r.get("description_text"),
        r.get("features_text"),
    ]),
    axis=1,
)
items_work["canonical_item_metadata_source_text"] = items_work["canonical_metadata_text"]

items_work["canonical_retrieval_text_core"] = items_work.apply(
    lambda r: join_unique_non_empty([r.get(col) for col in CORE_TEXT_SOURCE_COLUMNS], sep=" "),
    axis=1,
)
items_work["canonical_text_dense_core"] = items_work["canonical_retrieval_text_core"]
items_work["canonical_text_sparse_core"] = items_work.apply(
    lambda r: join_unique_non_empty([r.get(col) for col in CORE_SPARSE_TEXT_SOURCE_COLUMNS], sep=" "),
    axis=1,
)

# Initialize retrieval text from catalog evidence.
# Frozen review-derived item signals are appended after their pre-cutoff aggregation.
items_work["canonical_retrieval_text"] = items_work["canonical_retrieval_text_core"]
items_work["canonical_text_dense"] = items_work["canonical_text_dense_core"]
items_work["canonical_text_sparse"] = items_work["canonical_text_sparse_core"]

items_work["profile_safe_facet_text"] = items_work.apply(
    lambda row: join_unique_non_empty([
        row.get("brand_facet_text"),
        row.get("functional_facet_text"),
    ], sep=" "),
    axis=1,
)
items_work["profile_source_text_core"] = items_work.apply(
    lambda r: join_unique_non_empty([r.get(col) for col in PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS], sep=" "),
    axis=1,
)
items_work["profile_source_text_dedup_seed"] = items_work["profile_source_text_core"]

def contains_all_brand_values(container_text, brand_text):
    normalized_container = (clean_token_text(container_text) or "").lower()
    brand_values = split_facet_values(brand_text)
    return all(
        (clean_token_text(value) or "").lower() in normalized_container
        for value in brand_values
    )


brand_rows_mask = items_work["brand_facet_text"].fillna("").astype(str).str.strip().ne("")
brand_missing_retrieval_rows = int(
    items_work.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("canonical_retrieval_text_core"),
            row.get("brand_facet_text"),
        ),
        axis=1,
    ).sum()
)
brand_missing_profile_rows = int(
    items_work.loc[brand_rows_mask].apply(
        lambda row: not contains_all_brand_values(
            row.get("profile_safe_facet_text"),
            row.get("brand_facet_text"),
        ),
        axis=1,
    ).sum()
)
if brand_missing_retrieval_rows:
    raise RuntimeError(
        f"Brand values are missing from retrieval text for {brand_missing_retrieval_rows} items."
    )
if brand_missing_profile_rows:
    raise RuntimeError(
        f"Brand values are missing from profile-safe facets for {brand_missing_profile_rows} items."
    )

items_work["facet_policy_version"] = FACET_POLICY_VERSION
items_work["evidence_scope"] = PRODUCTION_EVIDENCE_SCOPE
items_work["historical_review_reputation_enabled"] = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED
items_work["historical_review_reputation_quarantined"] = HISTORICAL_REVIEW_REPUTATION_QUARANTINED
items_work["brand_policy"] = BRAND_POLICY
items_work["brand_retrieval_enabled"] = True
items_work["brand_profile_enabled"] = True
items_work["brand_query_enabled"] = False
items_work["identifier_policy"] = IDENTIFIER_POLICY

print("Rows:", len(items_work))
print("Validation: Global Review text contract initialized")
display(
    items_work[
        [
            "parent_asin",
            "title",
            "brand_facet_text",
            "query_safe_facet_text",
            "specific_query_safe_facet_text",
            "functional_facet_text",
            "canonical_retrieval_text_core",
            "canonical_retrieval_text",
            "profile_source_text_dedup_seed",
        ]
    ].head(5)
)


Rows: 27407
Validation: Global Review text contract initialized


,parent_asin,title,brand_facet_text,query_safe_facet_text,specific_query_safe_facet_text,functional_facet_text,canonical_retrieval_text_core,canonical_retrieval_text,profile_source_text_dedup_seed
0,B01LYS06EF,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),life extension,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,"Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) life extension curcumin capsule Healthy Inflammatory Vegetarian Adult [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioav...","Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) life extension curcumin capsule Healthy Inflammatory Vegetarian Adult [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioav...","life extension curcumin capsule Healthy Inflammatory Vegetarian Adult Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioav..."
1,B08T67YDQF,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces…",naturevibe botanicals,ginger | liquid | Adult,ginger | liquid | Adult,ginger | liquid | Adult,"Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces… naturevibe botanicals ginger liquid Adult [] [""The flavor is somewhat peppery and slightly sweet, with a strong and spicy aroma."", ""Apart from being a great flavoring agent, ginger root also makes a great daily supplement."", ""The Ginger...","Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces… naturevibe botanicals ginger liquid Adult [] [""The flavor is somewhat peppery and slightly sweet, with a strong and spicy aroma."", ""Apart from being a great flavoring agent, ginger root also makes a great daily supplement."", ""The Ginger...","naturevibe botanicals ginger liquid Adult Naturevibe Botanicals Ginger Root Extract Liquid, 2 Ounces… []"
2,B002LIMQQA,"Nature's Way Artichoke, 60 Capsules (Pack of 2)",nature's way,artichoke | capsule | Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* | Adult,artichoke | capsule | Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* | Adult,artichoke | capsule | Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* | Adult,"Nature's Way Artichoke, 60 Capsules (Pack of 2) nature's way artichoke capsule Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* Adult []","Nature's Way Artichoke, 60 Capsules (Pack of 2) nature's way artichoke capsule Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* Adult []","nature's way artichoke capsule Artichoke extract is standardized to 13-18% caffeoylquinic acids. Supports digestion.* Adult Nature's Way Artichoke, 60 Capsules (Pack of 2) []"
3,B0B1236V1Q,"Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free",mushroom revival,mushrooms | drops | Reishi Calm | Adult,mushrooms | drops | Reishi Calm | Adult,mushrooms | drops | Reishi Calm | Adult,"Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free mushroom revival mushrooms drops Reishi Calm Adult [] [""RELAX with the power of Reishi. Feel like a zen monk smiling throughout your day. Deflect occassional stress with your ow...","Mushroom Revival, Organic, Calm Reishi Tincture, 2 Fluid Ounces, Non-GMO, Vegan, Kosher, Gluten Free, Keto, Dairy Free mushroom revival mushrooms drops Reishi Calm Adult [] [""RELAX

In [14]:
# ==== Extract Frozen Review-Derived Item Signals ====
# Legacy internal names containing "review_reputation" denote frozen review-derived item signals.
# These item-level signals exclude target or future reviews, ratings, sentiment, raw-text exports, and user-profile use.


def normalize_review_space(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def review_timestamp_ms(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000
    return numeric


if "parent_asin" not in reviews.columns:
    raise RuntimeError("Raw Herbal reviews must contain parent_asin for historical reputation aggregation.")

review_timestamp_source_col = next(
    (col for col in ["review_datetime", "review_timestamp_ms", "timestamp", "unixReviewTime", "date"] if col in reviews.columns),
    None,
)
if review_timestamp_source_col is None:
    raise RuntimeError("No review timestamp column was found for historical review reputation.")

review_title_source_col = next(
    (col for col in ["title", "summary", "review_title"] if col in reviews.columns),
    None,
)
review_text_source_col = next(
    (col for col in ["text", "review_text", "review_body"] if col in reviews.columns),
    None,
)

if review_title_source_col is None and review_text_source_col is None:
    raise RuntimeError("No review title/body text column was found for historical review reputation.")

raw_review_rows_loaded_for_reputation = int(len(reviews))

review_reputation_df = pd.DataFrame({
    "parent_asin": reviews["parent_asin"].fillna("").astype(str).str.strip(),
    "review_timestamp_ms": review_timestamp_ms(reviews[review_timestamp_source_col]),
})
review_reputation_df["review_datetime"] = pd.to_datetime(
    review_reputation_df["review_timestamp_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

review_title_source = (
    reviews[review_title_source_col].fillna("").astype(str).map(normalize_review_space)
    if review_title_source_col
    else pd.Series("", index=reviews.index, dtype="object")
)
review_text_source = (
    reviews[review_text_source_col].fillna("").astype(str).map(normalize_review_space)
    if review_text_source_col
    else pd.Series("", index=reviews.index, dtype="object")
)
review_reputation_df["review_reputation_source_text"] = (
    review_title_source + " " + review_text_source
).map(normalize_review_space)

catalog_item_ids = set(items_work["parent_asin"].fillna("").astype(str).str.strip())
valid_review_mask = (
    review_reputation_df["parent_asin"].ne("")
    & review_reputation_df["review_datetime"].notna()
    & review_reputation_df["review_reputation_source_text"].ne("")
)
valid_reviews_for_reputation = review_reputation_df.loc[valid_review_mask].copy()
reviews_with_invalid_reputation_inputs = int((~valid_review_mask).sum())
reviews_outside_item_catalog = int((~valid_reviews_for_reputation["parent_asin"].isin(catalog_item_ids)).sum())
valid_reviews_for_reputation = valid_reviews_for_reputation[
    valid_reviews_for_reputation["parent_asin"].isin(catalog_item_ids)
].copy()

historical_review_mask = valid_reviews_for_reputation["review_datetime"] < TRAIN_REVIEW_CUTOFF_EXCLUSIVE
historical_reviews_for_reputation = valid_reviews_for_reputation.loc[historical_review_mask].copy()
catalog_overlap_item_count = int(
    historical_reviews_for_reputation["parent_asin"].nunique()
)
reviews_excluded_by_reputation_cutoff = int((~historical_review_mask).sum())

REVIEW_REPUTATION_SIGNAL_PATTERNS = {
    "historical_review_ingredient": {
        "turmeric": [r"\bturmeric\b"],
        "curcumin": [r"\bcurcumin\b"],
        "ginger": [r"\bginger\b"],
        "elderberry": [r"\belderberry\b"],
        "ashwagandha": [r"\bashwagandha\b"],
        "mushroom": [r"\bmushrooms?\b"],
        "lion's mane": [r"\blion'?s\s+mane\b"],
        "reishi": [r"\breishi\b"],
        "chaga": [r"\bchaga\b"],
        "milk thistle": [r"\bmilk\s+thistle\b"],
        "echinacea": [r"\bechinacea\b"],
        "ginseng": [r"\bginseng\b"],
        "cranberry": [r"\bcranberry\b"],
        "peppermint": [r"\bpeppermint\b"],
        "chamomile": [r"\bchamomile\b"],
        "valerian": [r"\bvalerian\b"],
        "garlic": [r"\bgarlic\b"],
        "berberine": [r"\bberberine\b"],
        "maca": [r"\bmaca\b"],
        "moringa": [r"\bmoringa\b"],
        "saw palmetto": [r"\bsaw\s+palmetto\b"],
        "black seed": [r"\bblack\s+seed\b"],
        "olive leaf": [r"\bolive\s+leaf\b"],
        "dong quai": [r"\bdong\s+quai\b"],
        "st john's wort": [r"\bst\.?\s*john'?s\s+wort\b"],
        "ginkgo biloba": [r"\bginkgo\s+biloba\b"],
        "holy basil": [r"\bholy\s+basil\b"],
        "evening primrose": [r"\bevening\s+primrose\b"],
        "aloe vera": [r"\baloe\s+vera\b"],
        "horny goat weed": [r"\bhorny\s+goat\s+weed\b"],
        "red yeast rice": [r"\bred\s+yeast\s+rice\b"],
    },
    "historical_review_benefit": {
        "immune": [r"\bimmune\b", r"\bimmunity\b"],
        "sleep": [r"\bsleep\b"],
        "stress": [r"\bstress\b"],
        "calm": [r"\bcalm\w*\b", r"\brelax\w*\b"],
        "digestion": [r"\bdigest\w*\b", r"\bgut\b", r"\bstomach\b"],
        "energy": [r"\benergy\b"],
        "focus": [r"\bfocus\b", r"\bconcentrat\w*\b"],
        "memory": [r"\bmemory\b"],
        "joint": [r"\bjoint\b"],
        "inflammation": [r"\binflamm\w*\b"],
        "liver": [r"\bliver\b"],
        "detox": [r"\bdetox\w*\b"],
        "throat": [r"\bthroat\b"],
        "urinary": [r"\burinary\b", r"\bbladder\b"],
        "heart": [r"\bheart\b", r"\bcardiovascular\b"],
        "blood sugar": [r"\bblood\s+sugar\b", r"\bglucose\b"],
        "mood": [r"\bmood\b"],
        "respiratory": [r"\brespiratory\b", r"\bbreathing\b"],
        "kidney": [r"\bkidney\b"],
        "vision": [r"\bvision\b", r"\beyes?\b"],
        "bone": [r"\bbones?\b"],
        "muscle": [r"\bmuscles?\b"],
        "circulation": [r"\bcirculation\b"],
        "cholesterol": [r"\bcholesterol\b"],
        "menopause": [r"\bmenopaus\w*\b", r"\bhormonal\b"],
        "prostate": [r"\bprostate\b"],
    },
    "historical_review_form": {
        "capsule": [r"\bcapsules?\b"],
        "tablet": [r"\btablets?\b"],
        "softgel": [r"\bsoftgels?\b"],
        "gummy": [r"\bgumm(?:y|ies)\b"],
        "powder": [r"\bpowder\b"],
        "tea": [r"\btea\b"],
        "liquid": [r"\bliquid\b"],
        "liquid extract": [r"\bliquid\s+extract\b"],
        "tincture": [r"\btincture\b"],
        "drops": [r"\bdrops?\b"],
        "syrup": [r"\bsyrup\b"],
        "spray": [r"\bspray\b"],
    },
    "historical_review_claim_diet": {
        "organic": [r"\borganic\b"],
        "vegan": [r"\bvegan\b"],
        "vegetarian": [r"\bvegetarian\b"],
        "non-gmo": [r"\bnon[- ]?gmo\b"],
        "gluten-free": [r"\bgluten[- ]?free\b"],
        "sugar-free": [r"\bsugar[- ]?free\b"],
        "caffeine-free": [r"\bcaffeine[- ]?free\b"],
        "dairy-free": [r"\bdairy[- ]?free\b"],
        "keto": [r"\bketo\b"],
        "kosher": [r"\bkosher\b"],
        "halal": [r"\bhalal\b"],
    },
    "historical_review_flavor": {
        "unflavored": [r"\bunflavou?red\b"],
        "berry": [r"\bberry\b"],
        "lemon": [r"\blemon\b"],
        "orange": [r"\borange\b"],
        "mint": [r"\bmint\b"],
        "cherry": [r"\bcherry\b"],
        "strawberry": [r"\bstrawberry\b"],
        "chocolate": [r"\bchocolate\b"],
        "vanilla": [r"\bvanilla\b"],
        "grape": [r"\bgrape\b"],
        "peach": [r"\bpeach\b"],
    },
}

REVIEW_REPUTATION_FAMILY_LABELS = {
    "historical_review_ingredient": "review ingredients",
    "historical_review_benefit": "review benefits",
    "historical_review_form": "review forms",
    "historical_review_claim_diet": "review claims",
    "historical_review_flavor": "review flavors",
}

compiled_review_reputation_patterns = {
    family: {
        label: re.compile("|".join(f"(?:{pattern})" for pattern in patterns), flags=re.IGNORECASE)
        for label, patterns in label_patterns.items()
    }
    for family, label_patterns in REVIEW_REPUTATION_SIGNAL_PATTERNS.items()
}


def extract_review_reputation_signals(text):
    source = normalize_review_space(text)
    output = {}
    for family, label_patterns in compiled_review_reputation_patterns.items():
        output[family] = [
            label for label, pattern in label_patterns.items()
            if pattern.search(source)
        ]
    return output


family_signal_counters = {family: Counter() for family in REVIEW_REPUTATION_SIGNAL_PATTERNS}
item_signal_counters = defaultdict(
    lambda: {family: Counter() for family in REVIEW_REPUTATION_SIGNAL_PATTERNS}
)
item_review_counts = historical_reviews_for_reputation.groupby("parent_asin").size().astype(int)

for parent_asin, source_text in historical_reviews_for_reputation[
    ["parent_asin", "review_reputation_source_text"]
].itertuples(index=False):
    extracted = extract_review_reputation_signals(source_text)
    for family, labels in extracted.items():
        labels = sorted(set(labels))
        family_signal_counters[family].update(labels)
        item_signal_counters[parent_asin][family].update(labels)

reputation_rows = []
# Keep every item with at least one historical review, even when no controlled signal is extracted.
for parent_asin in item_review_counts.index:
    family_counters = item_signal_counters[parent_asin]
    row = {
        "parent_asin": parent_asin,
        "historical_review_count_pre_cutoff": int(item_review_counts.get(parent_asin, 0)),
    }
    text_parts = []
    signal_family_count = 0
    signal_total_count = 0

    for family, family_label in REVIEW_REPUTATION_FAMILY_LABELS.items():
        top_labels = [
            label for label, _ in family_counters[family].most_common(REVIEW_REPUTATION_MAX_PHRASES_PER_FAMILY)
        ]
        family_text = " | ".join(top_labels)
        row[f"{family}_text"] = family_text
        if top_labels:
            signal_family_count += 1
            signal_total_count += len(top_labels)
            text_parts.append(f"{family_label}: {', '.join(top_labels)}")

    row["historical_review_reputation_text"] = " | ".join(text_parts)
    row["review_reputation_signal_family_count"] = int(signal_family_count)
    row["review_reputation_signal_total_count"] = int(signal_total_count)
    row["review_reputation_available_flag"] = bool(signal_total_count > 0)
    reputation_rows.append(row)

review_reputation_features_df = pd.DataFrame(reputation_rows)
review_reputation_cols = [
    "historical_review_count_pre_cutoff",
    "historical_review_reputation_text",
    "historical_review_ingredient_text",
    "historical_review_benefit_text",
    "historical_review_form_text",
    "historical_review_claim_diet_text",
    "historical_review_flavor_text",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "review_reputation_available_flag",
]
if review_reputation_features_df.empty:
    review_reputation_features_df = pd.DataFrame(columns=["parent_asin"] + review_reputation_cols)

review_reputation_existing_cols = [
    col for col in items_work.columns
    if col.startswith("historical_review_")
    or col.startswith("review_reputation_")
    or col == "common_review_derived_signal_text"
]
items_work = items_work.drop(columns=review_reputation_existing_cols, errors="ignore")
items_work["parent_asin"] = items_work["parent_asin"].fillna("").astype(str).str.strip()
items_work = items_work.merge(
    review_reputation_features_df,
    on="parent_asin",
    how="left",
    validate="one_to_one",
)

review_reputation_text_cols = [
    "historical_review_reputation_text",
    "historical_review_ingredient_text",
    "historical_review_benefit_text",
    "historical_review_form_text",
    "historical_review_claim_diet_text",
    "historical_review_flavor_text",
]
for column in review_reputation_text_cols:
    items_work[column] = items_work[column].fillna("").map(normalize_review_space)
for column in [
    "historical_review_count_pre_cutoff",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
]:
    items_work[column] = items_work[column].fillna(0).astype(int)
items_work["review_reputation_available_flag"] = (
    items_work["review_reputation_available_flag"].astype("boolean").fillna(False).astype(bool)
)

items_work["review_reputation_facet_text"] = items_work["historical_review_reputation_text"]
items_work["review_reputation_source"] = REVIEW_REPUTATION_SOURCE
items_work["review_reputation_cutoff_exclusive"] = TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat()

# Restore production policy flags after dropping/rebuilding review-reputation columns.
items_work["historical_review_reputation_enabled"] = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED
items_work["historical_review_reputation_quarantined"] = HISTORICAL_REVIEW_REPUTATION_QUARANTINED

assert "historical_review_reputation_enabled" in items_work.columns
assert "historical_review_reputation_quarantined" in items_work.columns

if historical_reviews_for_reputation["review_datetime"].ge(TRAIN_REVIEW_CUTOFF_EXCLUSIVE).any():
    raise RuntimeError("Historical review reputation includes a review on or after the training cutoff.")
if RATING_USED_FOR_REVIEW_REPUTATION or SENTIMENT_USED_FOR_REVIEW_REPUTATION or LLM_USED_FOR_REVIEW_REPUTATION:
    raise RuntimeError("Historical review reputation must not use rating, sentiment, or LLM-derived signals.")
if RAW_REVIEW_TEXT_EXPORTED_FOR_REVIEW_REPUTATION:
    raise RuntimeError("Raw review text must not be exported as a review-reputation feature.")
for forbidden_column in ["review_reputation_source_text", "raw_review_text", "target_review_text"]:
    if forbidden_column in items_work.columns:
        raise RuntimeError(f"Raw review text column was exported to item schema: {forbidden_column}")

items_with_review_reputation = int(items_work["review_reputation_available_flag"].sum())
review_reputation_item_coverage = float(items_work["review_reputation_available_flag"].mean())
review_reputation_diag_rows = [
    {"metric": "raw_review_rows_loaded", "family": "overall", "value": raw_review_rows_loaded_for_reputation},
    {"metric": "invalid_or_empty_review_rows_excluded", "family": "overall", "value": reviews_with_invalid_reputation_inputs},
    {"metric": "reviews_outside_item_catalog_excluded", "family": "overall", "value": reviews_outside_item_catalog},
    {"metric": "historical_reviews_before_cutoff", "family": "overall", "value": int(len(historical_reviews_for_reputation))},
    {"metric": "reviews_on_or_after_cutoff_excluded", "family": "overall", "value": reviews_excluded_by_reputation_cutoff},
    {"metric": "items_with_review_reputation", "family": "overall", "value": items_with_review_reputation},
    {"metric": "review_reputation_item_coverage", "family": "overall", "value": review_reputation_item_coverage},
    {"metric": "average_historical_review_count_pre_cutoff", "family": "overall", "value": float(items_work["historical_review_count_pre_cutoff"].mean())},
]
for family in REVIEW_REPUTATION_SIGNAL_PATTERNS:
    family_col = f"{family}_text"
    review_reputation_diag_rows.append({
        "metric": "non_empty_rate",
        "family": family,
        "value": float(items_work[family_col].fillna("").astype(str).str.strip().ne("").mean()),
    })
    for rank, (signal, count) in enumerate(family_signal_counters[family].most_common(20), start=1):
        review_reputation_diag_rows.append({
            "metric": f"top_signal_{rank:02d}:{signal}",
            "family": family,
            "value": int(count),
        })

review_reputation_diagnostics_df = pd.DataFrame(review_reputation_diag_rows)
review_reputation_diagnostics_df.to_csv(
    OUTPUT_DIR / "herbal_item_review_reputation_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)

# Rebuild the merged retrieval text after adding frozen review-derived item signals.
items_work["review_reputation_only_text"] = (
    items_work["review_reputation_facet_text"]
    .fillna("")
    .astype(str)
    .map(normalize_review_space)
)
items_work["canonical_retrieval_text"] = items_work.apply(
    lambda row: join_unique_non_empty([
        row.get("canonical_retrieval_text_core"),
        row.get("review_reputation_only_text"),
    ], sep=" "),
    axis=1,
)
items_work["canonical_text_dense"] = items_work["canonical_retrieval_text"]
items_work["canonical_text_sparse"] = items_work.apply(
    lambda row: join_unique_non_empty([
        row.get("canonical_text_sparse_core"),
        row.get("review_reputation_only_text"),
    ], sep=" "),
    axis=1,
)
# User-profile source text remains catalog-only; frozen review-derived signals are excluded.
items_work["profile_source_text_dedup_seed"] = items_work["profile_source_text_core"]

items_work["evidence_scope"] = PRODUCTION_EVIDENCE_SCOPE
items_work["historical_review_reputation_enabled"] = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED
items_work["historical_review_reputation_quarantined"] = HISTORICAL_REVIEW_REPUTATION_QUARANTINED

if not items_work["review_reputation_only_text"].fillna("").astype(str).str.strip().ne("").any():
    raise RuntimeError("Production Global Review requires non-empty historical review signals.")
print("Historical review reputation source:", REVIEW_REPUTATION_SOURCE)
print("Training cutoff exclusive:", TRAIN_REVIEW_CUTOFF_EXCLUSIVE)
print("Historical review rows:", len(historical_reviews_for_reputation))
print("Catalog overlap items:", catalog_overlap_item_count)
print("Items with review reputation:", items_with_review_reputation)
print("Review reputation item coverage:", round(review_reputation_item_coverage, 4))
print("Validation: passed")

# Release temporary row-level text frames before later item-level aggregation.
del review_reputation_df
del valid_reviews_for_reputation
del historical_reviews_for_reputation


Historical review reputation source: historical_reviews_before_train_cutoff
Training cutoff exclusive: 2022-03-31 23:56:10.358000+00:00
Historical review rows: 498333
Catalog overlap items: 17196
Items with review reputation: 12624
Review reputation item coverage: 0.4606
Validation: passed


In [15]:
# ==== Aggregate Review Diagnostics per Parent Item ====
reviews_work = reviews.copy()

# Prepare temporary fields for diagnostic aggregation
reviews_work["text"] = reviews_work["text"].fillna("").astype(str)
reviews_work["title"] = reviews_work["title"].fillna("").astype(str)

# Convert diagnostic fields to numeric form
reviews_work["rating_num"] = pd.to_numeric(reviews_work["rating"], errors="coerce")
reviews_work["verified_purchase_num"] = (
    reviews_work["verified_purchase"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": 1, "false": 0})
)

reviews_work["helpful_vote_num"] = pd.to_numeric(reviews_work["helpful_vote"], errors="coerce")
reviews_work["review_text_len"] = reviews_work["text"].astype(str).str.len()

print("reviews_work shape:", reviews_work.shape)
print("rating_num nulls:", reviews_work["rating_num"].isna().sum())
print("verified_purchase_num nulls:", reviews_work["verified_purchase_num"].isna().sum())

review_item_summary = (
    reviews_work
    .groupby("parent_asin")
    .agg(
        review_count=("parent_asin", "size"),
        review_user_count=("user_id", "nunique"),
        avg_review_rating=("rating_num", "mean"),
        verified_purchase_ratio=("verified_purchase_num", "mean"),
        mean_review_text_len=("review_text_len", "mean"),
        mean_helpful_vote=("helpful_vote_num", "mean"),
    )
    .reset_index()
)

print("review_item_summary shape:", review_item_summary.shape)
display(review_item_summary.head(5))

reviews_work shape: (593363, 13)
rating_num nulls: 0
verified_purchase_num nulls: 0
review_item_summary shape: (19622, 7)


,parent_asin,review_count,review_user_count,avg_review_rating,verified_purchase_ratio,mean_review_text_len,mean_helpful_vote
0,0006446469,4,4,3.75,1.0,190.0,0.75
1,0922658277,2,2,5.00,1.0,90.0,0.00
2,0929619730,1,1,5.00,1.0,93.0,0.00
3,3597126197,1,1,1.00,1.0,50.0,0.00
4,3780378507,5,5,3.40,1.0,133.0,0.60


In [16]:
# ==== Attach Review Diagnostics to the Intermediate Schema ====
item_schema_base = items_work.merge(
    review_item_summary,
    on="parent_asin",
    how="left",
)

item_schema_base["review_reputation_only_text"] = (
    item_schema_base["review_reputation_facet_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)
item_schema_base["common_review_derived_signal_text"] = item_schema_base["review_reputation_only_text"]

item_schema_base["canonical_retrieval_text"] = item_schema_base.apply(
    lambda r: join_unique_non_empty([
        r.get("canonical_retrieval_text_core"),
        r.get("review_reputation_only_text"),
    ], sep=" "),
    axis=1,
)
item_schema_base["canonical_text_dense"] = item_schema_base["canonical_retrieval_text"]
item_schema_base["canonical_text_sparse"] = item_schema_base.apply(
    lambda r: join_unique_non_empty([
        r.get("canonical_text_sparse_core"),
        r.get("review_reputation_only_text"),
    ], sep=" "),
    axis=1,
)
item_schema_base["profile_source_text_dedup_seed"] = item_schema_base["profile_source_text_core"]

print("Rows:", len(item_schema_base))
print("Validation: Global Review item representation assembled")


Rows: 27407
Validation: Global Review item representation assembled


In [17]:
# ==== Remove Unused Raw Metadata Fields ====
DROP_COLS = [
    "main_category",
    "price",
    "bought_together",
    "brand_from_details",
    "details_parsed",
    "categories_parsed",
]

item_schema_base = item_schema_base.drop(columns=[c for c in DROP_COLS if c in item_schema_base.columns])

print("item_schema_base shape after drop:", item_schema_base.shape)

item_schema_base shape after drop: (27407, 106)


In [18]:
# ==== Organize the Intermediate Schema ====
front_cols = [
    "parent_asin",
    "title",
    "brand_primary",
    "brand_norm",
    "sub_category",
    "sub_category_norm",
    "average_rating",
    "rating_number",
    "review_count",
    "review_user_count",
    "avg_review_rating",
    "verified_purchase_ratio",
    "category_path_text",
    "category_depth",
    "item_form",
    "item_form_norm",
    "age_range",
    "recommended_use",
    "specific_uses",
    "product_benefits",
    "active_ingredients",
    "special_ingredients",
    "primary_supplement_type",
    "diet_type",
    "material_feature",
    "flavor",
    "scent",
    "manufacturer",
    "date_first_available",
    "is_discontinued",
    "is_discontinued_norm",
    "family_brand",
    "family_sub_category",
    "family_item_form",
    "family_benefit_function",
    "family_usage_need",
    "family_ingredient_composition",
    "family_claims_diet",
    "family_flavor",
    "family_usage_target",
    "brand_facet_text",
    "facet_brand_text",
    "facet_category_text",
    "facet_form_text",
    "facet_ingredient_text",
    "facet_benefit_text",
    "facet_claim_text",
    "facet_flavor_text",
    "facet_audience_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "identifier_diagnostic_text",
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
    "facet_policy_version",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "historical_review_reputation_quarantined",
    "brand_policy",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "identifier_policy",
    "features",
    "description",
    "details",
    "categories",
    "historical_review_reputation_text",
    "review_reputation_only_text",
    "common_review_derived_signal_text",
]

front_cols = [c for c in front_cols if c in item_schema_base.columns]
other_cols = [c for c in item_schema_base.columns if c not in front_cols]
item_schema_base = item_schema_base[front_cols + other_cols].copy()

display(item_schema_base.head(5))


,parent_asin,title,brand_primary,brand_norm,sub_category,sub_category_norm,average_rating,rating_number,review_count,review_user_count,avg_review_rating,verified_purchase_ratio,category_path_text,category_depth,item_form,item_form_norm,age_range,recommended_use,specific_uses,product_benefits,active_ingredients,special_ingredients,primary_supplement_type,diet_type,material_feature,flavor,scent,manufacturer,date_first_available,is_discontinued,is_discontinued_norm,family_brand,family_sub_category,family_item_form,family_benefit_function,family_usage_need,family_ingredient_composition,family_claims_diet,family_flavor,family_usage_target,brand_facet_text,facet_brand_text,facet_category_text,facet_form_text,facet_ingredient_text,facet_benefit_text,facet_claim_text,facet_flavor_text,facet_audience_text,query_safe_facet_text,specific_query_safe_facet_text,functional_facet_text,review_reputation_facet_text,identifier_diagnostic_text,canonical_metadata_text,canonical_item_metadata_source_text,canonical_retrieval_text_core,canonical_text_dense_core,canonical_text_sparse_core,canonical_retrieval_text,canonical_text_dense,canonical_text_sparse,profile_safe_facet_text,profile_source_text_core,profile_source_text_dedup_seed,facet_policy_version,evidence_scope,historical_review_reputation_enabled,historical_review_reputation_quarantined,brand_policy,brand_retrieval_enabled,brand_profile_enabled,brand_query_enabled,identifier_policy,features,description,details,categories,historical_review_reputation_text,review_reputation_only_text,common_review_derived_signal_text,store,unit_count,number_of_items,package_dimensions,item_model_number,brand,title_text,features_text,description_text,generic_category_anchor_text,generic_utility_token_text,context_dependent_utility_token_text,historical_review_count_pre_cutoff,historical_review_ingredient_text,historical_review_benefit_text,historical_review_form_text,historical_review_claim_diet_text,historical_review_flavor_text,review_reputation_signal_family_count,review_reputation_signal_total_count,review_reputation_available_flag,review_reputation_source,review_reputation_cutoff_exclusive,mean_review_text_len,mean_helpful_vote
0,B01LYS06EF,Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack),Life Extension,life extension,Curcumin,curcumin,4.6,375,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Curcumin",4,capsules,capsule,Adult,Healthy Inflammatory,None,None,None,None,None,None,Vegetarian,None,None,Life Extension,"April 10, 2014",No,no,life extension,curcumin,capsule,None,Healthy Inflammatory,None,Vegetarian,None,Adult,life extension,life extension,curcumin,capsule,,Healthy Inflammatory,Vegetarian,,Adult,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult,,B01LYS06EF | Life Extension | 00407,"Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) | life extension | curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult | [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produ...","Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) | life extension | curcumin | capsule | Healthy Inflammatory | Vegetarian | Adult | [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produ...","Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) life extension curcumin capsule Healthy Inflammatory Vegetarian Adult [""Life Extension Bio-Curcumin Elite protects free curcuminoids from conjugation by combining curcumin with fenugreek fibers. Doing so produces more bioav...","Life Extension Bio-Curcumin Elite 400 mg 60 Vegetarian Capsules (2 Pack) life extension curcumin capsule Healthy Inflammator

In [19]:
# ==== Check and Remove Duplicate Columns ====
duplicate_cols = item_schema_base.columns[item_schema_base.columns.duplicated()].tolist()
print("Duplicate columns before cleanup:", duplicate_cols)

if duplicate_cols:
    item_schema_base = item_schema_base.loc[:, ~item_schema_base.columns.duplicated()].copy()

duplicate_cols = item_schema_base.columns[item_schema_base.columns.duplicated()].tolist()
print("Duplicate columns after cleanup:", duplicate_cols)

Duplicate columns before cleanup: []
Duplicate columns after cleanup: []


In [20]:
# ==== Summarize Column Coverage ====
column_summary = pd.DataFrame({
    "column": item_schema_base.columns,
    "dtype": item_schema_base.dtypes.astype(str).values,
    "non_null_count": item_schema_base.notna().sum().values,
    "null_count": item_schema_base.isna().sum().values,
    "null_ratio": item_schema_base.isna().mean().values,
    "nunique": [safe_nunique(item_schema_base[c]) for c in item_schema_base.columns],
}).sort_values(["null_ratio", "nunique"], ascending=[False, False])

display(column_summary.head(100))

,column,dtype,non_null_count,null_count,null_ratio,nunique
26,scent,object,127,27280,0.995366,60
18,specific_uses,object,157,27250,0.994272,107
20,active_ingredients,object,192,27215,0.992994,145
22,primary_supplement_type,object,314,27093,0.988543,116
21,special_ingredients,object,1746,25661,0.936294,570
...,...,...,...,...,...,...
65,facet_policy_version,object,27407,0,0.000000,1
66,evidence_scope,object,27407,0,0.000000,1
67,historical_review_reputation_enabled,bool,27407,0,0.000000,1
68,historical_review_reputation_quarantined,bool,27407,0,0.000000,1


In [21]:
# ==== Verify Column Uniqueness ====
duplicate_cols = item_schema_base.columns[item_schema_base.columns.duplicated()].tolist()
print("Duplicate columns:", duplicate_cols)

if duplicate_cols:
    for col in sorted(set(duplicate_cols)):
        positions = [i for i, c in enumerate(item_schema_base.columns) if c == col]
        print(col, positions)

Duplicate columns: []


In [22]:
# ==== Summarize Facet-Family Coverage ====
major_feature_cols = [
    "family_brand",
    "family_sub_category",
    "family_item_form",
    "family_benefit_function",
    "family_usage_need",
    "family_ingredient_composition",
    "family_claims_diet",
    "family_flavor",
    "family_usage_target",
    "brand_facet_text",
    "facet_brand_text",
    "facet_category_text",
    "facet_form_text",
    "facet_ingredient_text",
    "facet_benefit_text",
    "facet_claim_text",
    "facet_flavor_text",
    "facet_audience_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
]
major_feature_cols = [c for c in major_feature_cols if c in item_schema_base.columns]

coverage_summary = pd.DataFrame({
    "column": major_feature_cols,
    "non_null_count": [item_schema_base[c].notna().sum() for c in major_feature_cols],
    "coverage_ratio": [item_schema_base[c].notna().mean() for c in major_feature_cols],
    "nunique": [safe_nunique(item_schema_base[c]) for c in major_feature_cols],
}).sort_values("coverage_ratio", ascending=False)

display(coverage_summary)


,column,non_null_count,coverage_ratio,nunique
1,family_sub_category,27407,1.000000,74
13,facet_ingredient_text,27407,1.000000,751
10,facet_brand_text,27407,1.000000,6468
11,facet_category_text,27407,1.000000,75
12,facet_form_text,27407,1.000000,503
9,brand_facet_text,27407,1.000000,6468
21,canonical_retrieval_text_core,27407,1.000000,26984
20,functional_facet_text,27407,1.000000,12156
19,specific_query_safe_facet_text,27407,1.000000,12156
18,query_safe_facet_text,27407,1.000000,13051


In [23]:
# ==== Validate Downstream Evidence Contracts ====
qc_cols = [
    "parent_asin",
    "title",
    "brand_primary",
    "sub_category",
    "item_form",
    "recommended_use",
    "specific_uses",
    "product_benefits",
    "active_ingredients",
    "special_ingredients",
    "diet_type",
    "flavor",
    "family_brand",
    "family_sub_category",
    "family_item_form",
    "family_benefit_function",
    "family_usage_need",
    "family_ingredient_composition",
    "family_claims_diet",
    "family_flavor",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "canonical_retrieval_text_core",
    "canonical_retrieval_text",
    "profile_source_text_dedup_seed",
]
qc_cols = [c for c in qc_cols if c in item_schema_base.columns]

DOWNSTREAM_REQUIRED_SCHEMA_COLS = [
    "parent_asin",
    "title",
    "brand_facet_text",
    "facet_brand_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "identifier_diagnostic_text",
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
    "facet_policy_version",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "historical_review_reputation_quarantined",
    "brand_policy",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "identifier_policy",
    "facet_category_text",
    "facet_form_text",
    "facet_ingredient_text",
    "facet_benefit_text",
    "facet_claim_text",
    "facet_flavor_text",
    "facet_audience_text",
]

missing_downstream_cols = [c for c in DOWNSTREAM_REQUIRED_SCHEMA_COLS if c not in item_schema_base.columns]
if missing_downstream_cols:
    raise RuntimeError(f"Schema base is missing downstream-required columns: {missing_downstream_cols}")

if item_schema_base["parent_asin"].isna().any() or item_schema_base["parent_asin"].astype(str).str.strip().eq("").any():
    raise RuntimeError("parent_asin contains null or empty values.")
if not item_schema_base["parent_asin"].astype(str).is_unique:
    raise RuntimeError("parent_asin must be unique.")

for text_col in [
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "profile_source_text_dedup_seed",
]:
    empty_count = int(item_schema_base[text_col].fillna("").astype(str).str.strip().eq("").sum())
    if empty_count:
        raise RuntimeError(f"{text_col} must be non-empty for every item. Empty rows: {empty_count}")


for output_col, source_cols in SOURCE_LISTS_USED_FOR_PRODUCTION_CORE.items():
    forbidden_sources = [
        col for col in source_cols
        if col in HISTORICAL_REVIEW_TEXT_COLUMNS
        or col.startswith("historical_review_")
        or col.startswith("review_reputation_")
    ]
    if forbidden_sources:
        raise RuntimeError(f"Historical review sources feed {output_col}: {forbidden_sources}")

brand_sources = set(HARMONIZED_BRAND_SOURCE_COLS + ["facet_brand_text", "brand_facet_text"])
for output_col in ["query_safe_facet_text", "functional_facet_text"]:
    source_cols = set(SOURCE_LISTS_USED_FOR_PRODUCTION_CORE[output_col])
    if brand_sources.intersection(source_cols):
        raise RuntimeError(f"Brand source columns feed {output_col}.")
if "brand_facet_text" not in SOURCE_LISTS_USED_FOR_PRODUCTION_CORE["canonical_retrieval_text_core"]:
    raise RuntimeError("Brand must be included in production retrieval text.")
if "brand_facet_text" not in PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS:
    raise RuntimeError("Brand must be included in the profile source.")

if not PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED:
    raise RuntimeError("Historical review-reputation must be enabled for production Global Review.")
if not item_schema_base["review_reputation_only_text"].fillna("").astype(str).str.strip().ne("").any():
    raise RuntimeError("Production Global Review requires non-empty historical review signals.")

policy_checks = {
    "facet_policy_version": FACET_POLICY_VERSION,
    "brand_policy": BRAND_POLICY,
    "identifier_policy": IDENTIFIER_POLICY,
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
}
for policy_col, expected_value in policy_checks.items():
    observed = sorted(item_schema_base[policy_col].dropna().astype(str).unique().tolist())
    if observed != [expected_value]:
        raise RuntimeError(f"Unexpected {policy_col} values: {observed}")

print("Rows:", len(item_schema_base))
print("Validation: downstream Global Review contract checks passed")
display(item_schema_base[qc_cols].sample(min(20, len(item_schema_base)), random_state=42))


Rows: 27407
Validation: downstream Global Review contract checks passed


,parent_asin,title,brand_primary,sub_category,item_form,recommended_use,specific_uses,product_benefits,active_ingredients,special_ingredients,diet_type,flavor,family_brand,family_sub_category,family_item_form,family_benefit_function,family_usage_need,family_ingredient_composition,family_claims_diet,family_flavor,brand_facet_text,query_safe_facet_text,specific_query_safe_facet_text,functional_facet_text,canonical_retrieval_text_core,canonical_retrieval_text,profile_source_text_dedup_seed
7497,B002DNV2YA,Nature's Sunshine HistaBlock 90 Capsules (Pack of 2),Nature's Sunshine,Herbal Supplements,Capsule,None,None,None,None,None,None,None,nature's sunshine,herbal supplements,capsule,None,None,None,None,None,nature's sunshine,herbal supplements | capsule | Adult,herbal supplements | capsule | Adult,herbal supplements | capsule | Adult,Nature's Sunshine HistaBlock 90 Capsules (Pack of 2) nature's sunshine herbal supplements capsule Adult [],Nature's Sunshine HistaBlock 90 Capsules (Pack of 2) nature's sunshine herbal supplements capsule Adult [],nature's sunshine herbal supplements capsule Adult Nature's Sunshine HistaBlock 90 Capsules (Pack of 2) []
24825,B000XPG5SI,"Sundown Cranberry Fruit Capsules, 475mg, 100-Count Bottle",Sundown,Cranberry,Capsules,Urinary Tract Health,None,None,None,None,None,None,sundown,cranberry,capsule,None,Urinary Tract Health,None,Natural,None,sundown,cranberry | capsule | Urinary Tract Health | Natural | Adult,cranberry | capsule | Urinary Tract Health | Adult,cranberry | capsule | Urinary Tract Health | Adult,"Sundown Cranberry Fruit Capsules, 475mg, 100-Count Bottle sundown cranberry capsule Urinary Tract Health Natural Adult [""Product description"", ""Sundown Cranberry Fruit Capsule is a convenient way to get the full spectrum of cranberry‘s natural beneficial compounds. Benefits Healthy Urinary Funct...","Sundown Cranberry Fruit Capsules, 475mg, 100-Count Bottle sundown cranberry capsule Urinary Tract Health Natural Adult [""Product description"", ""Sundown Cranberry Fruit Capsule is a convenient way to get the full spectrum of cranberry‘s natural beneficial compounds. Benefits Healthy Urinary Funct...","sundown cranberry capsule Urinary Tract Health Adult Sundown Cranberry Fruit Capsules, 475mg, 100-Count Bottle [""Product description"", ""Sundown Cranberry Fruit Capsule is a convenient way to get the full spectrum of cranberry‘s natural beneficial compounds. Benefits Healthy Urinary Function. 100..."
13911,B09D2Y8MP9,BAI SHAO - 白芍 - White Peony Root - FUHENG福恒 - Since 1905-100g 1 Container Not Powdered,FUHENG,Alfalfa,None,None,None,None,None,None,None,None,fuheng,alfalfa,None,None,None,None,None,None,fuheng,alfalfa,alfalfa,alfalfa,BAI SHAO - 白芍 - White Peony Root - FUHENG福恒 - Since 1905-100g 1 Container Not Powdered fuheng alfalfa [],BAI SHAO - 白芍 - White Peony Root - FUHENG福恒 - Since 1905-100g 1 Container Not Powdered fuheng alfalfa [],fuheng alfalfa BAI SHAO - 白芍 - White Peony Root - FUHENG福恒 - Since 1905-100g 1 Container Not Powdered []
705,B07QJZDWNP,"Terry Naturally Clinical OPC 150 mg - 60 Vegan Capsules - French Grape Seed Extract Supplement - Antioxidant - Non-GMO, Gluten Free - 60 Servings",Terry Naturally,Grape Seed Extract,Capsules,Antioxidant,None,None,None,None,None,None,terry naturally,grape seed extract,capsule,None,Antioxidant,None,None,None,terry naturally,grape seed extract | capsule | Antioxidant | Adult,grape seed extract | capsule | Antioxidant | Adult,grape seed extract | capsule | Antioxidant | Adult,"Terry Naturally Clinical OPC 150 mg - 60 Vegan Capsules - French Grape Seed Extract Supplement - Antioxidant - Non-GMO, Gluten Free - 60 Servings terry naturally grape seed extract capsule Antioxidant Adult [] [""Superior Absorption for Full Benefits; Research has found the benefits of grape seed...","Terry Naturally Clinical OPC 150 mg - 60 Vegan Capsules - French Grape Seed Extract Supplement - Antioxidant - Non-GMO, Gluten Free - 60 Servings terry naturally grape seed extract 

In [24]:
# ==== Export Canonical Item-Schema Artifacts ====
item_schema_base.to_parquet(SCHEMA_BASE_PATH, index=False)
column_summary.to_csv(COLUMN_SUMMARY_PATH, index=False, encoding="utf-8-sig")
coverage_summary.to_csv(COVERAGE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
item_schema_base[qc_cols].head(int(CONFIG["qc_preview_rows"])).to_csv(QC_PREVIEW_PATH, index=False, encoding="utf-8-sig")

created_outputs = [
    SCHEMA_BASE_PATH,
    COLUMN_SUMMARY_PATH,
    COVERAGE_SUMMARY_PATH,
    QC_PREVIEW_PATH,
]

results_overall = pd.DataFrame([
    {"metric": "items_input_rows", "value": len(items)},
    {"metric": "reviews_input_rows", "value": len(reviews)},
    {"metric": "schema_base_rows", "value": len(item_schema_base)},
    {"metric": "schema_base_columns", "value": item_schema_base.shape[1]},
    {"metric": "unique_parent_asin", "value": item_schema_base["parent_asin"].nunique()},
    {"metric": "duplicate_parent_asin_rows", "value": int(item_schema_base["parent_asin"].duplicated().sum())},
])
results_overall.to_csv(RESULTS_OVERALL_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(RESULTS_OVERALL_PATH)

diagnostics_summary = pd.DataFrame([
    {"check": "items_path_exists", "value": ITEMS_PATH.exists()},
    {"check": "reviews_path_exists", "value": REVIEWS_PATH.exists()},
    {"check": "schema_base_has_rows", "value": len(item_schema_base) > 0},
    {"check": "parent_asin_unique", "value": item_schema_base["parent_asin"].is_unique},
    {"check": "column_summary_rows", "value": len(column_summary)},
    {"check": "coverage_summary_rows", "value": len(coverage_summary)},
    {"check": "downstream_required_schema_columns_present", "value": len(missing_downstream_cols) == 0},
    {"check": "production_evidence_scope_global_review", "value": PRODUCTION_EVIDENCE_SCOPE == "catalog_metadata_functional_facets_and_historical_review_signals"},
    {"check": "historical_review_reputation_enabled_for_production", "value": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED},
    {"check": "canonical_retrieval_text_core_non_empty", "value": item_schema_base["canonical_retrieval_text_core"].fillna("").astype(str).str.strip().ne("").all()},
    {"check": "canonical_text_dense_core_non_empty", "value": item_schema_base["canonical_text_dense_core"].fillna("").astype(str).str.strip().ne("").all()},
    {"check": "canonical_text_sparse_core_non_empty", "value": item_schema_base["canonical_text_sparse_core"].fillna("").astype(str).str.strip().ne("").all()},
    {"check": "profile_source_text_dedup_seed_matches_profile_source", "value": item_schema_base["profile_source_text_dedup_seed"].fillna("").astype(str).equals(item_schema_base["profile_source_text_core"].fillna("").astype(str))},
    {"check": "brand_in_retrieval_enabled", "value": item_schema_base["brand_retrieval_enabled"].eq(True).all()},
    {"check": "brand_in_profile_enabled", "value": item_schema_base["brand_profile_enabled"].eq(True).all()},
    {"check": "brand_in_query_disabled", "value": item_schema_base["brand_query_enabled"].eq(False).all()},
    {"check": "facet_policy_version_expected", "value": item_schema_base["facet_policy_version"].eq(FACET_POLICY_VERSION).all()},
    {"check": "brand_policy_expected", "value": item_schema_base["brand_policy"].eq(BRAND_POLICY).all()},
    {"check": "identifier_policy_expected", "value": item_schema_base["identifier_policy"].eq(IDENTIFIER_POLICY).all()},
])
diagnostics_summary.to_csv(DIAGNOSTICS_SUMMARY_PATH, index=False, encoding="utf-8-sig")
created_outputs.append(DIAGNOSTICS_SUMMARY_PATH)

print("Output:", SCHEMA_BASE_PATH)
print("Rows:", len(item_schema_base))
print("Validation: canonical Global Review outputs saved")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/items_Herbal_Supplements_W2_2019_2022_schema_base.parquet
Rows: 27407
Validation: canonical Global Review outputs saved


## Cross-Category Semantic-Role Mapping

Category-specific metadata fields are mapped to shared semantic roles for later cross-category diagnostics. This mapping standardizes analytical roles without forcing the two categories to share vocabularies, thresholds, or coverage. It preserves the distinction among catalog-functional facets, Brand, and frozen review-derived item signals.


### Brand as a Separate Preference Facet

Brand is retained in retrieval and profile-safe representations and exported as its own facet family. It is excluded from synthetic-query evidence and from product-functional facet metrics. Brand-history association can therefore be examined without treating Brand as a functional product characteristic.


In [25]:
# ==== Map Category Fields to Shared Semantic Roles ====
COMMON_SCHEMA_POLICY_VERSION = "common_schema_roles_v6_global_review_brand_retrieval_profile"
IDENTIFIER_POLICY_COMMON = "diagnostic_only"
BRAND_POLICY_COMMON = "Brand is retained as a separate query-unsafe preference facet and is active in retrieval, graph, and user-profile representations."
REVIEW_SIGNAL_POLICY_COMMON = "historical_review_reputation_used_as_global_item_side_retrieval_evidence_only"
GENERIC_ANCHOR_POLICY_COMMON = "generic category anchors are tracked separately from specific facet cues in query-generation diagnostics"
FUNCTIONAL_FACET_DEFINITION_COMMON = "Specific product/preference facets excluding brand, generic category anchors, standalone generic utility tokens, context-dependent standalone utility tokens, review-derived rows, and diagnostic/count fields."
GENERIC_UTILITY_TOKENS_COMMON = HERBAL_GENERIC_UTILITY_TOKENS
CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON = HERBAL_CONTEXT_DEPENDENT_UTILITY_TOKENS
SPECIFIC_PHRASE_EXCEPTIONS_COMMON = HERBAL_SPECIFIC_PHRASE_EXCEPTIONS
HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS = {
    "milk", "olive", "saw", "seed", "dong", "john", "biloba", "holy",
    "evening", "horny", "vera", "red", "black", "fruit",
}
BRAND_DIAGNOSTIC_METRICS = [
    "brand_overlap",
    "brand_match",
    "seen_brand_match",
    "profile_brand_overlap",
    "weighted_facet_overlap_with_brand",
    "weighted_facet_overlap_no_brand",
    "brand_share_of_overlap",
]

COMMON_SCHEMA_ROLES = [
    "brand",
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
    "review_derived_signal",
]
COMMON_QUERY_SAFE_ROLES = [
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
COMMON_PROFILE_ROLES = ["brand"] + COMMON_QUERY_SAFE_ROLES

GENERIC_CATEGORY_ANCHORS_COMMON = HERBAL_GENERIC_CATEGORY_ANCHORS
ROLE_SOURCE_MAP = {
    "brand": ["brand_facet_text", "facet_brand_text", "family_brand", "brand_primary", "brand_norm", "brand"],
    "category_or_product_type": ["facet_category_text", "family_sub_category", "sub_category_norm", "sub_category"],
    "form_texture": ["facet_form_text", "family_item_form", "item_form_norm", "item_form"],
    "ingredient_or_composition": ["facet_ingredient_text", "family_ingredient_composition", "active_ingredients", "special_ingredients"],
    "need_benefit_concern": ["facet_benefit_text", "family_benefit_function", "family_usage_need", "product_benefits", "specific_uses"],
    "claim_constraint": ["facet_claim_text", "family_claims_diet", "material_feature", "material_type_free"],
    "target_context": ["facet_audience_text", "family_usage_target", "age_range", "recommended_uses"],
    "sensory": ["facet_flavor_text", "family_flavor", "flavor"],
    "review_derived_signal": [
        "review_reputation_facet_text",
        "historical_review_reputation_text",
        "historical_review_ingredient_text",
        "historical_review_benefit_text",
        "historical_review_form_text",
        "historical_review_claim_diet_text",
        "historical_review_flavor_text",
    ],
}

if "item_schema_norm" in globals():
    item_schema_common_df = item_schema_norm.copy()
elif "item_schema_base" in globals():
    item_schema_common_df = item_schema_base.copy()
else:
    raise RuntimeError("Expected item_schema_norm or item_schema_base before common schema role mapping.")

if "parent_asin" not in item_schema_common_df.columns:
    raise RuntimeError("Common schema framework requires parent_asin.")
item_schema_common_df["parent_asin"] = item_schema_common_df["parent_asin"].astype(str).str.strip()
if item_schema_common_df["parent_asin"].isna().any() or item_schema_common_df["parent_asin"].eq("").any():
    raise RuntimeError("parent_asin contains null or empty values.")
if not item_schema_common_df["parent_asin"].is_unique:
    raise RuntimeError("parent_asin must be unique before saving common schema outputs.")


def _common_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return " | ".join(_common_text(v) for v in value if _common_text(v))
    if isinstance(value, dict):
        return " | ".join(_common_text(v) for v in value.values() if _common_text(v))
    text = str(value).replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if text.lower() in {"", "none", "null", "nan", "n/a", "na", "[]", "{}"}:
        return ""
    return text


def _common_split(value):
    text = _common_text(value)
    if not text:
        return []
    out = []
    for part in re.split(r"\s*\|\s*|\s*;\s*", text):
        part = _common_text(part)
        if part:
            out.append(part)
    return out


def _common_join(values):
    out = []
    seen = set()
    for value in values:
        for part in _common_split(value):
            key = part.lower()
            if key and key not in seen:
                seen.add(key)
                out.append(part)
    return " | ".join(out)


def _join_existing_columns(row, columns):
    return _common_join(row[col] for col in columns if col in row.index)


role_rows = []
for role in COMMON_SCHEMA_ROLES:
    source_cols = ROLE_SOURCE_MAP[role]
    available_cols = [col for col in source_cols if col in item_schema_common_df.columns]
    missing_cols = [col for col in source_cols if col not in item_schema_common_df.columns]
    out_col = f"common_{role}_text"
    item_schema_common_df[out_col] = item_schema_common_df.apply(
        lambda row, cols=available_cols: _join_existing_columns(row, cols),
        axis=1,
    )
    role_rows.append({
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "common_role": role,
        "output_column": out_col,
        "source_columns_requested": " | ".join(source_cols),
        "source_columns_available": " | ".join(available_cols),
        "source_columns_missing": " | ".join(missing_cols),
        "query_safe_role": role in COMMON_QUERY_SAFE_ROLES,
        "profile_role": role in COMMON_PROFILE_ROLES,
        "retrieval_role": role in COMMON_QUERY_SAFE_ROLES or role in {"brand", "review_derived_signal"},
        "brand_role": role == "brand",
        "product_functional_role": role in COMMON_QUERY_SAFE_ROLES,
        "review_derived_role": role == "review_derived_signal",
        "policy_version": COMMON_SCHEMA_POLICY_VERSION,
    })

common_schema_role_map = pd.DataFrame(role_rows)
query_safe_common_cols = [f"common_{role}_text" for role in COMMON_QUERY_SAFE_ROLES]
profile_common_cols = ["common_brand_text"] + query_safe_common_cols

item_schema_common_df["common_query_safe_facet_text"] = item_schema_common_df.apply(
    lambda row: _join_existing_columns(row, query_safe_common_cols),
    axis=1,
)

_generic_anchor_norms = {anchor.lower() for anchor in GENERIC_CATEGORY_ANCHORS_COMMON}
_generic_utility_norms = {value.lower() for value in GENERIC_UTILITY_TOKENS_COMMON}
_context_utility_norms = {value.lower() for value in CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON}
_specific_exception_norms = {value.lower() for value in SPECIFIC_PHRASE_EXCEPTIONS_COMMON}


def _partition_common_query_safe_text(value):
    generic_anchors = []
    generic_utilities = []
    context_utilities = []
    specific_values = []

    for part in _common_split(value):
        norm = part.lower()
        if norm in _specific_exception_norms:
            specific_values.append(part)
        elif norm in _generic_anchor_norms:
            generic_anchors.append(part)
        elif norm in _generic_utility_norms:
            generic_utilities.append(part)
        elif norm in _context_utility_norms:
            context_utilities.append(part)
        else:
            specific_values.append(part)

    return pd.Series({
        "common_generic_anchor_text": _common_join(generic_anchors),
        "common_generic_utility_text": _common_join(generic_utilities),
        "common_context_dependent_utility_text": _common_join(context_utilities),
        "common_specific_query_safe_facet_text": _common_join(specific_values),
    })


partition_cols = [
    "common_generic_anchor_text",
    "common_generic_utility_text",
    "common_context_dependent_utility_text",
    "common_specific_query_safe_facet_text",
]
item_schema_common_df = item_schema_common_df.drop(columns=partition_cols, errors="ignore")
item_schema_common_df = pd.concat(
    [item_schema_common_df, item_schema_common_df["common_query_safe_facet_text"].apply(_partition_common_query_safe_text)],
    axis=1,
)
item_schema_common_df["common_brand_facet_text"] = item_schema_common_df["common_brand_text"].fillna("").astype(str).map(_common_text)
item_schema_common_df["common_functional_facet_text"] = item_schema_common_df["common_specific_query_safe_facet_text"]
item_schema_common_df["common_profile_safe_facet_text"] = item_schema_common_df.apply(
    lambda row: _common_join([
        row.get("common_brand_facet_text"),
        row.get("common_functional_facet_text"),
    ]),
    axis=1,
)
item_schema_common_df["common_profile_schema_text"] = item_schema_common_df["common_profile_safe_facet_text"]
item_schema_common_df["common_canonical_metadata_text"] = item_schema_common_df["canonical_metadata_text"]
item_schema_common_df["common_canonical_retrieval_text_core"] = item_schema_common_df["canonical_retrieval_text_core"]
item_schema_common_df["common_canonical_text_dense_core"] = item_schema_common_df["canonical_text_dense_core"]
item_schema_common_df["common_canonical_text_sparse_core"] = item_schema_common_df["canonical_text_sparse_core"]
item_schema_common_df["common_review_reputation_only_text"] = item_schema_common_df["review_reputation_only_text"]
item_schema_common_df["common_canonical_retrieval_text"] = item_schema_common_df["canonical_retrieval_text"]
item_schema_common_df["common_canonical_text_dense"] = item_schema_common_df["canonical_text_dense"]
item_schema_common_df["common_canonical_text_sparse"] = item_schema_common_df["canonical_text_sparse"]
item_schema_common_df["common_profile_source_text_core"] = item_schema_common_df["profile_source_text_core"]
item_schema_common_df["common_profile_source_text_dedup_seed"] = item_schema_common_df["profile_source_text_dedup_seed"]
item_schema_common_df["common_schema_policy_version"] = COMMON_SCHEMA_POLICY_VERSION
item_schema_common_df["common_identifier_policy"] = IDENTIFIER_POLICY_COMMON
item_schema_common_df["common_brand_policy"] = BRAND_POLICY_COMMON
item_schema_common_df["common_review_signal_policy"] = REVIEW_SIGNAL_POLICY_COMMON
item_schema_common_df["common_generic_anchor_policy"] = GENERIC_ANCHOR_POLICY_COMMON
item_schema_common_df["common_evidence_scope"] = PRODUCTION_EVIDENCE_SCOPE
item_schema_common_df["common_historical_review_reputation_enabled"] = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED


coverage_rows = []
for role in COMMON_SCHEMA_ROLES:
    col = f"common_{role}_text"
    non_empty = item_schema_common_df[col].fillna("").astype(str).str.strip().ne("")
    values = []
    for value in item_schema_common_df.loc[non_empty, col]:
        values.extend([v.lower() for v in _common_split(value)])
    coverage_rows.append({
        "category_id": CATEGORY_ID,
        "category_folder": CATEGORY_FOLDER,
        "category_label": CATEGORY_LABEL,
        "common_role": role,
        "column": col,
        "n_items": len(item_schema_common_df),
        "non_empty_items": int(non_empty.sum()),
        "coverage_rate": float(non_empty.mean()),
        "distinct_value_count": int(len(set(values))),
        "query_safe_role": role in COMMON_QUERY_SAFE_ROLES,
        "profile_role": role in COMMON_PROFILE_ROLES,
    })
common_schema_role_coverage = pd.DataFrame(coverage_rows)

core_uniformity_roles = [
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
]
coverage_values = common_schema_role_coverage.loc[
    common_schema_role_coverage["common_role"].isin(core_uniformity_roles),
    "coverage_rate",
].astype(float)
common_schema_uniformity_summary = pd.DataFrame([{
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "core_roles": " | ".join(core_uniformity_roles),
    "core_coverage_mean": float(coverage_values.mean()) if len(coverage_values) else np.nan,
    "core_coverage_min": float(coverage_values.min()) if len(coverage_values) else np.nan,
    "core_coverage_max": float(coverage_values.max()) if len(coverage_values) else np.nan,
    "core_coverage_range": float(coverage_values.max() - coverage_values.min()) if len(coverage_values) else np.nan,
    "core_coverage_std": float(coverage_values.std(ddof=0)) if len(coverage_values) else np.nan,
    "core_coverage_cv": float(coverage_values.std(ddof=0) / coverage_values.mean()) if len(coverage_values) and coverage_values.mean() != 0 else np.nan,
}])

facet_long_rows = []
for _, row in item_schema_common_df.iterrows():
    item_id = row["parent_asin"]
    for role in COMMON_SCHEMA_ROLES:
        role_col = f"common_{role}_text"
        for value in _common_split(row.get(role_col)):
            norm = value.lower()
            is_brand = role == "brand"
            is_review_derived = role == "review_derived_signal"
            is_generic_anchor = norm in _generic_anchor_norms
            is_generic_utility = norm in _generic_utility_norms
            is_context_utility = norm in _context_utility_norms
            is_exception = norm in _specific_exception_norms
            is_specific = (
                role in COMMON_QUERY_SAFE_ROLES
                and not is_brand
                and not is_review_derived
                and (
                    is_exception
                    or (
                        not is_generic_anchor
                        and not is_generic_utility
                        and not is_context_utility
                    )
                )
            )
            facet_long_rows.append({
                "parent_asin": item_id,
                "category_id": CATEGORY_ID,
                "category_folder": CATEGORY_FOLDER,
                "category_label": CATEGORY_LABEL,
                "common_role": role,
                "facet_role": role,
                "facet_family": role,
                "facet_value": value,
                "facet_value_norm": norm,
                "is_brand": is_brand,
                "is_review_derived": is_review_derived,
                "is_generic_category_anchor": is_generic_anchor,
                "is_generic_utility_token": is_generic_utility,
                "is_context_dependent_utility_token": is_context_utility,
                "is_specific_phrase_exception": is_exception,
                "is_profile_role": role in COMMON_PROFILE_ROLES,
                "is_retrieval_safe": bool(is_brand or is_specific or is_review_derived),
                "is_profile_safe": bool(is_brand or is_specific),
                "policy_version": COMMON_SCHEMA_POLICY_VERSION,
            })

items_facets_common = pd.DataFrame(facet_long_rows)
if items_facets_common.empty:
    raise RuntimeError("Common item facet long table is empty.")

items_facets_common["is_possible_entity_fragment"] = (
    items_facets_common["facet_role"].eq("ingredient_or_composition")
    & items_facets_common["facet_value_norm"].fillna("").astype(str).str.lower().isin(HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS)
)
items_facets_common["is_specific_facet_phrase"] = (
    items_facets_common["facet_role"].isin(COMMON_QUERY_SAFE_ROLES)
    & ~items_facets_common["is_brand"]
    & ~items_facets_common["is_review_derived"]
    & (
        items_facets_common["is_specific_phrase_exception"]
        | (
            ~items_facets_common["is_generic_category_anchor"]
            & ~items_facets_common["is_generic_utility_token"]
            & ~items_facets_common["is_context_dependent_utility_token"]
        )
    )
)
items_facets_common["is_product_functional_facet"] = items_facets_common["is_specific_facet_phrase"]
items_facets_common["is_query_safe"] = items_facets_common["is_product_functional_facet"]

facet_vocab_common = (
    items_facets_common
    .groupby(["category_id", "common_role", "facet_value_norm"], as_index=False)
    .agg(
        facet_value=("facet_value", "first"),
        item_count=("parent_asin", "nunique"),
        is_query_safe=("is_query_safe", "max"),
        is_profile_role=("is_profile_role", "max"),
        is_retrieval_safe=("is_retrieval_safe", "max"),
        is_profile_safe=("is_profile_safe", "max"),
        is_brand=("is_brand", "max"),
        is_product_functional_facet=("is_product_functional_facet", "max"),
        is_generic_category_anchor=("is_generic_category_anchor", "max"),
        is_generic_utility_token=("is_generic_utility_token", "max"),
        is_context_dependent_utility_token=("is_context_dependent_utility_token", "max"),
        is_specific_facet_phrase=("is_specific_facet_phrase", "max"),
        is_possible_entity_fragment=("is_possible_entity_fragment", "max"),
        is_review_derived=("is_review_derived", "max"),
    )
    .sort_values(["common_role", "item_count", "facet_value_norm"], ascending=[True, False, True])
)

brand_rows = items_facets_common[items_facets_common["is_brand"]].copy()
functional_rows = items_facets_common[items_facets_common["is_product_functional_facet"]].copy()
role_coverage_with_brand = common_schema_role_coverage.copy()
role_coverage_without_brand = common_schema_role_coverage.loc[~common_schema_role_coverage["common_role"].eq("brand")].copy()
role_coverage_without_brand.to_csv(COMMON_SCHEMA_COVERAGE_NO_BRAND_PATH, index=False, encoding="utf-8-sig")

functional_coverage_values = role_coverage_without_brand.loc[
    role_coverage_without_brand["common_role"].isin(COMMON_QUERY_SAFE_ROLES),
    "coverage_rate",
].astype(float)
coverage_uniformity_excluding_brand = float(functional_coverage_values.std(ddof=0)) if len(functional_coverage_values) else np.nan

brand_policy_summary = pd.DataFrame([
    {"metric": "brand_coverage_rate", "value": float(item_schema_common_df["common_brand_text"].fillna("").astype(str).str.strip().ne("").mean())},
    {"metric": "functional_facet_coverage_rate", "value": float(functional_rows["parent_asin"].nunique() / len(item_schema_common_df)) if len(item_schema_common_df) else np.nan},
    {"metric": "brand_unique_value_count", "value": int(brand_rows["facet_value_norm"].nunique())},
    {"metric": "functional_facet_unique_value_count", "value": int(functional_rows["facet_value_norm"].nunique())},
    {"metric": "role_coverage_with_brand", "value": float(role_coverage_with_brand["coverage_rate"].mean())},
    {"metric": "role_coverage_without_brand", "value": float(role_coverage_without_brand["coverage_rate"].mean())},
    {"metric": "coverage_uniformity_excluding_brand", "value": coverage_uniformity_excluding_brand},
])
brand_policy_summary.to_csv(BRAND_POLICY_SUMMARY_PATH, index=False, encoding="utf-8-sig")

common_schema_diagnostics = pd.DataFrame([
    {"check": "common_role_map_created", "value": len(common_schema_role_map) == len(COMMON_SCHEMA_ROLES)},
    {"check": "common_item_schema_has_rows", "value": len(item_schema_common_df) > 0},
    {"check": "common_parent_asin_unique", "value": item_schema_common_df["parent_asin"].is_unique},
    {"check": "production_evidence_scope_global_review", "value": PRODUCTION_EVIDENCE_SCOPE == "catalog_metadata_functional_facets_and_historical_review_signals"},
    {"check": "historical_review_reputation_enabled_for_production", "value": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED},
    {"check": "common_query_safe_facet_text_non_empty_rate", "value": float(item_schema_common_df["common_query_safe_facet_text"].fillna("").astype(str).str.strip().ne("").mean())},
    {"check": "common_profile_source_text_non_empty_rate", "value": float(item_schema_common_df["common_profile_source_text_dedup_seed"].fillna("").astype(str).str.strip().ne("").mean())},
    {"check": "common_facet_long_rows", "value": int(len(items_facets_common))},
    {"check": "generic_anchor_policy_recorded", "value": True},
    {"check": "brand_excluded_from_query_safe_policy", "value": bool((~items_facets_common.loc[items_facets_common["is_brand"], "is_query_safe"]).all())},
    {"check": "brand_retrieval_safe", "value": bool(items_facets_common.loc[items_facets_common["is_brand"], "is_retrieval_safe"].all())},
    {"check": "brand_profile_safe", "value": bool(items_facets_common.loc[items_facets_common["is_brand"], "is_profile_safe"].all())},
    {"check": "brand_not_product_functional_facet", "value": bool((~items_facets_common.loc[items_facets_common["is_brand"], "is_product_functional_facet"]).all())},
    {"check": "review_derived_not_query_safe", "value": bool((~items_facets_common.loc[items_facets_common["is_review_derived"], "is_query_safe"]).all())},
    {"check": "review_derived_not_product_functional", "value": bool((~items_facets_common.loc[items_facets_common["is_review_derived"], "is_product_functional_facet"]).all())},
    {"check": "generic_anchor_not_specific_facet_phrase", "value": bool((~items_facets_common.loc[items_facets_common["is_generic_category_anchor"], "is_specific_facet_phrase"]).all())},
    {"check": "generic_utility_not_specific_facet_phrase", "value": bool((~items_facets_common.loc[items_facets_common["is_generic_utility_token"], "is_specific_facet_phrase"]).all())},
    {"check": "context_utility_not_specific_without_exception", "value": bool((~items_facets_common.loc[items_facets_common["is_context_dependent_utility_token"] & ~items_facets_common["is_specific_phrase_exception"], "is_specific_facet_phrase"]).all())},
    {"check": "possible_entity_fragment_warning_rows", "value": int(items_facets_common["is_possible_entity_fragment"].sum())},
    {"check": "identifier_diagnostic_only_policy", "value": True},
])

common_schema_role_map.to_csv(COMMON_SCHEMA_ROLE_MAP_PATH, index=False, encoding="utf-8-sig")
common_schema_role_coverage.to_csv(COMMON_SCHEMA_COVERAGE_PATH, index=False, encoding="utf-8-sig")
common_schema_uniformity_summary.to_csv(COMMON_SCHEMA_UNIFORMITY_PATH, index=False, encoding="utf-8-sig")
common_schema_diagnostics.to_csv(COMMON_SCHEMA_DIAGNOSTICS_PATH, index=False, encoding="utf-8-sig")

item_schema_common_df.to_parquet(COMMON_ITEM_SCHEMA_PATH, index=False)
items_facets_common.to_parquet(ITEM_FACETS_PATH, index=False)
facet_vocab_common.to_parquet(FACET_VOCAB_PATH, index=False)
item_schema_common_df.to_parquet(SCHEMA_OUTPUT_PATH, index=False)
item_schema_common_df.to_parquet(SCHEMA_FULL_OUTPUT_PATH, index=False)
if "SCHEMA_BASE_PATH" in globals():
    item_schema_common_df.to_parquet(SCHEMA_BASE_PATH, index=False)
if "SCHEMA_NORM_OUTPUT_PATH" in globals():
    item_schema_common_df.to_parquet(SCHEMA_NORM_OUTPUT_PATH, index=False)

item_schema_norm = item_schema_common_df
item_schema_base = item_schema_common_df

common_schema_contract = {
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "policy_version": COMMON_SCHEMA_POLICY_VERSION,
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED,
    "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
    "project_root": str(PROJECT_ROOT),
    "items_path": str(ITEMS_PATH),
    "reviews_path": str(REVIEWS_PATH),
    "common_roles": COMMON_SCHEMA_ROLES,
    "common_query_safe_roles": COMMON_QUERY_SAFE_ROLES,
    "common_profile_roles": COMMON_PROFILE_ROLES,
    "generic_category_anchors": sorted(GENERIC_CATEGORY_ANCHORS_COMMON),
    "generic_utility_tokens": sorted(GENERIC_UTILITY_TOKENS_COMMON),
    "context_dependent_utility_tokens": sorted(CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON),
    "specific_phrase_exceptions": sorted(SPECIFIC_PHRASE_EXCEPTIONS_COMMON),
    "possible_multiword_entity_fragment_tokens": sorted(HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS),
    "generic_anchor_policy": GENERIC_ANCHOR_POLICY_COMMON,
    "query_audit_note": (
        "Standalone support/routine/formula-like carriers are not specific cues. "
        "Benefit phrases such as immune support remain specific. "
        "Fragment flags are warnings only and must be checked against source entity spans."
    ),
    "brand_policy": BRAND_POLICY_COMMON,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
    "functional_facet_definition": FUNCTIONAL_FACET_DEFINITION_COMMON,
    "specific_facet_phrase_policy": (
        "Specific facet phrases exclude brand, generic category anchors, standalone generic utility "
        "tokens, review-derived rows, and non-functional roles. Phrase-level values such as immune "
        "support remain specific because only standalone utility-token values are flagged as generic "
        "utility tokens."
    ),
    "brand_diagnostic_metrics": BRAND_DIAGNOSTIC_METRICS,
    "identifier_policy": IDENTIFIER_POLICY_COMMON,
    "review_signal_policy": REVIEW_SIGNAL_POLICY_COMMON,
    "role_source_map": ROLE_SOURCE_MAP,
    "outputs": {
        "common_item_schema": str(COMMON_ITEM_SCHEMA_PATH),
        "canonical_schema_output": str(SCHEMA_OUTPUT_PATH),
        "schema_full_output": str(SCHEMA_FULL_OUTPUT_PATH),
        "items_facets": str(ITEM_FACETS_PATH),
        "facet_vocab": str(FACET_VOCAB_PATH),
        "role_map": str(COMMON_SCHEMA_ROLE_MAP_PATH),
        "role_coverage": str(COMMON_SCHEMA_COVERAGE_PATH),
        "uniformity_summary": str(COMMON_SCHEMA_UNIFORMITY_PATH),
        "brand_policy_summary": str(BRAND_POLICY_SUMMARY_PATH),
        "role_coverage_no_brand": str(COMMON_SCHEMA_COVERAGE_NO_BRAND_PATH),
        "diagnostics": str(COMMON_SCHEMA_DIAGNOSTICS_PATH),
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open(COMMON_SCHEMA_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(common_schema_contract, f, ensure_ascii=False, indent=2)

common_created_outputs = [
    COMMON_ITEM_SCHEMA_PATH,
    ITEM_FACETS_PATH,
    FACET_VOCAB_PATH,
    COMMON_SCHEMA_ROLE_MAP_PATH,
    COMMON_SCHEMA_COVERAGE_PATH,
    COMMON_SCHEMA_UNIFORMITY_PATH,
    COMMON_SCHEMA_DIAGNOSTICS_PATH,
    COMMON_SCHEMA_CONTRACT_PATH,
    BRAND_POLICY_SUMMARY_PATH,
    COMMON_SCHEMA_COVERAGE_NO_BRAND_PATH,
]
if "created_outputs" in globals():
    created_outputs.extend([p for p in common_created_outputs if p not in created_outputs])

if "CONFIG" in globals() and isinstance(CONFIG, dict):
    CONFIG["common_schema_policy_version"] = COMMON_SCHEMA_POLICY_VERSION
    CONFIG["common_schema_contract_path"] = str(COMMON_SCHEMA_CONTRACT_PATH)
    CONFIG["common_item_schema_path"] = str(COMMON_ITEM_SCHEMA_PATH)
    CONFIG["item_facets_path"] = str(ITEM_FACETS_PATH)
    CONFIG["facet_vocab_path"] = str(FACET_VOCAB_PATH)

print("Output:", COMMON_ITEM_SCHEMA_PATH)
print("Rows:", len(item_schema_common_df))
print("Validation: common Global Review schema export passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema_common.parquet
Rows: 27407
Validation: common Global Review schema export passed


In [26]:
# ==== Reload and Validate Shared-Schema Artifacts ====
common_required_cols = [
    "common_brand_text",
    "common_category_or_product_type_text",
    "common_form_texture_text",
    "common_ingredient_or_composition_text",
    "common_need_benefit_concern_text",
    "common_claim_constraint_text",
    "common_target_context_text",
    "common_sensory_text",
    "common_review_derived_signal_text",
    "common_query_safe_facet_text",
    "common_generic_anchor_text",
    "common_generic_utility_text",
    "common_context_dependent_utility_text",
    "common_specific_query_safe_facet_text",
    "common_brand_facet_text",
    "common_functional_facet_text",
    "common_profile_safe_facet_text",
    "common_profile_schema_text",
    "common_canonical_metadata_text",
    "common_canonical_retrieval_text_core",
    "common_canonical_text_dense_core",
    "common_canonical_text_sparse_core",
    "common_review_reputation_only_text",
    "common_canonical_retrieval_text",
    "common_canonical_text_dense",
    "common_canonical_text_sparse",
    "common_profile_source_text_core",
    "common_profile_source_text_dedup_seed",
    "common_schema_policy_version",
    "common_evidence_scope",
    "common_historical_review_reputation_enabled",
]

for path in [COMMON_ITEM_SCHEMA_PATH, globals().get("SCHEMA_OUTPUT_PATH", COMMON_ITEM_SCHEMA_PATH)]:
    if not path:
        continue
    df_reload = pd.read_parquet(path)
    missing_common_cols = sorted(set(common_required_cols) - set(df_reload.columns))
    print("Output:", path)
    print("Rows:", len(df_reload))
    print("Validation: common schema columns checked")
    if missing_common_cols:
        raise RuntimeError(f"Missing common schema columns in {path}: {missing_common_cols}")

facets_reload = pd.read_parquet(ITEM_FACETS_PATH)
vocab_reload = pd.read_parquet(FACET_VOCAB_PATH)
print("Output:", ITEM_FACETS_PATH)
print("Rows:", len(facets_reload))
print("Output:", FACET_VOCAB_PATH)
print("Rows:", len(vocab_reload))

if facets_reload.empty:
    raise RuntimeError("Common item facets reload is empty.")
if vocab_reload.empty:
    raise RuntimeError("Common facet vocab reload is empty.")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema_common.parquet
Rows: 27407
Validation: common schema columns checked
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema.parquet
Rows: 27407
Validation: common schema columns checked
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_items_facets.parquet
Rows: 247480
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_facet_vocab.parquet
Rows: 20529


In [27]:
# ==== Write Provenance and Configuration Manifests ====
def jsonable_value(value):
    if isinstance(value, dict):
        return {str(k): jsonable_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable_value(v) for v in value]
    if isinstance(value, set):
        return sorted([jsonable_value(v) for v in value], key=lambda x: str(x))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def write_simple_yaml(data, output_path):
    def scalar(value):
        if value is None:
            return "null"
        if isinstance(value, bool):
            return "true" if value else "false"
        if isinstance(value, (int, float)):
            return str(value)
        text = str(value).replace('"', '\\"')
        return f'"{text}"'

    def emit(obj, indent=0):
        lines = []
        pad = " " * indent
        if isinstance(obj, dict):
            for key, value in obj.items():
                if isinstance(value, (dict, list)):
                    lines.append(f"{pad}{key}:")
                    lines.extend(emit(value, indent + 2))
                else:
                    lines.append(f"{pad}{key}: {scalar(value)}")
        elif isinstance(obj, list):
            for value in obj:
                if isinstance(value, (dict, list)):
                    lines.append(f"{pad}-")
                    lines.extend(emit(value, indent + 2))
                else:
                    lines.append(f"{pad}- {scalar(value)}")
        return lines

    Path(output_path).write_text("\n".join(emit(jsonable_value(data))) + "\n", encoding="utf-8")


manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "project_root": str(PROJECT_ROOT),
    "input_paths": {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items()},
    "created_outputs": [str(path) for path in created_outputs],
    "output_dir": str(OUTPUT_DIR),
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED,
    "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
    "downstream_contract": {
        "required_schema_columns": DOWNSTREAM_REQUIRED_SCHEMA_COLS,
        "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
        "historical_review_reputation_enabled": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED,
        "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
        "facet_policy_version": FACET_POLICY_VERSION,
        "brand_policy": BRAND_POLICY,
        "common_brand_policy": BRAND_POLICY_COMMON if "BRAND_POLICY_COMMON" in globals() else "",
        "brand_in_retrieval_text": True,
        "brand_in_profile_source_text": True,
        "brand_in_synthetic_query": False,
        "functional_facet_definition": FUNCTIONAL_FACET_DEFINITION_COMMON if "FUNCTIONAL_FACET_DEFINITION_COMMON" in globals() else "",
        "brand_diagnostic_metrics": BRAND_DIAGNOSTIC_METRICS if "BRAND_DIAGNOSTIC_METRICS" in globals() else [],
        "identifier_policy": IDENTIFIER_POLICY,
        "compatible_downstream_notebooks": [
            "03_user_regime_sampling_herbal.ipynb",
            "04_retrieval_artifact_herbal.ipynb",
            "05_review_signal_extraction_herbal.ipynb",
        ],
    },
    "summary": {
        "items_input_rows": int(len(items)),
        "reviews_input_rows": int(len(reviews)),
        "schema_base_rows": int(len(item_schema_base)),
        "schema_base_columns": int(item_schema_base.shape[1]),
        "unique_parent_asin": int(item_schema_base["parent_asin"].nunique()),
        "duplicate_parent_asin_rows": int(item_schema_base["parent_asin"].duplicated().sum()),
    },
    "environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
    },
}

with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(jsonable_value(manifest), f, ensure_ascii=False, indent=2)

write_simple_yaml(CONFIG, CONFIG_SNAPSHOT_PATH)
created_outputs.extend([RUN_MANIFEST_PATH, CONFIG_SNAPSHOT_PATH])

print("Output:", RUN_MANIFEST_PATH)
print("Rows:", len(item_schema_base))
print("Validation: manifest saved")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage0_feature_engineering/run_manifest.json
Rows: 27407
Validation: manifest saved


In [28]:
# ==== Reload and Validate the Production Item Schema ====
schema = pd.read_parquet(SCHEMA_OUTPUT_PATH)

required_core_cols = [
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "brand_facet_text",
    "profile_safe_facet_text",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "common_canonical_retrieval_text",
    "common_canonical_retrieval_text_core",
]
missing_core_cols = sorted(set(required_core_cols) - set(schema.columns))
if missing_core_cols:
    raise RuntimeError(f"Missing required production columns: {missing_core_cols}")

if schema["canonical_retrieval_text_core"].fillna("").astype(str).str.strip().eq("").any():
    raise RuntimeError("canonical_retrieval_text_core must be non-empty for every item.")
if not schema["brand_retrieval_enabled"].eq(True).all():
    raise RuntimeError("brand_retrieval_enabled must be True.")
if not schema["brand_profile_enabled"].eq(True).all():
    raise RuntimeError("brand_profile_enabled must be True.")
if not schema["brand_query_enabled"].eq(False).all():
    raise RuntimeError("brand_query_enabled must be False.")
if not schema["parent_asin"].astype(str).is_unique:
    raise RuntimeError("parent_asin must be unique.")

print("Output:", SCHEMA_OUTPUT_PATH)
print("Rows:", len(schema))
print("Validation: production Global Review reload passed")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/items/herbal_item_schema.parquet
Rows: 27407
Validation: production Global Review reload passed
